# Seed-specific character experiment

Portable Python 3.12 source snapshot. Configure inputs and enable the final run cell explicitly.


## Environment

Use the locked project environment or install the matching dependencies before running. NumPy 2.1.3 and pypinyin 0.55.0 are required. Audio inspection needs soundfile; source export needs datasets. Real inference/training additionally needs Torch 2.11.0 and the corresponding optional dependencies. No dependency installation is performed by this notebook.


In [ ]:
from pathlib import Path
import hashlib
import sys
import tempfile

if any(name == "phonetics" or name.startswith("phonetics.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before loading this source snapshot")
SDK_ROOT = Path(tempfile.mkdtemp(prefix="phonetics-sdk-"))
SOURCE_HASHES = {'__init__.py': 'fef444ae8ce0352410108db2801f88bbec818fecf7774aa996ceca99ba72f437', '__main__.py': 'e7824f0f3c4ca4641aeeb4a5436d552d3bf24e3f594b34fa0710e37a87b295b7', 'cli.py': '74fc12c8cd010269b0ce2caaad7ff6a1b0ac6e1f2788b16a69b427b7a85143d0', 'pipeline.py': '553fa3adf77c2553d7524d0195ea8699d1b70718bc19b4cdb3ccbc775a564769', 'artifacts.py': '9447013a45a347b7df6735cde9dc7c40f0931d53792165999c482c384321d64d', 'bundles.py': '0c4f85e7d71ce1604d35d60a766c03945c5ace79fa7e40a7ef6f1315ee3f549d', 'data_contract.py': 'c412fe15925d8052d56e15daa2fb0d855bc6a62f47aa3ac7a74983e1a411621b', 'pinyin.py': '068f53115ff99936d22177444a8b545b51195e188d0bd69230c930cd26976d67', 'objectives.py': '2cdb95c3044103e6138d28038d696e43b200ea7b0e94c3131cac9a3fd3c0df47', 'sound_inventory.py': 'fa0b065ef3d5e164192b5de3b308e5acb6522bceac1a4ba64e7761fcc79b47b7', 'corpus.py': 'e69b487fb77baece5f05e5d0c360747ae62777c7e13440f92fcc9326a871ffc2', 'acoustic.py': 'c8e60fc549f9619de8839d8f4e85e12582bf1fd6c075dbb98c872fcacc8e18b2', 'experiment.py': 'a95424768341001e222feebc1c7faa6af9fbbf02bd05eb1a87016a076a4633d4', 'model.py': 'c6c2fbf0cbaad28ecbf889ef83b990fa18d197b90576c9bfcc279f76227fb134', 'data.py': '48fdc7c66924bbbd26cc9603e240f5355ffa9e240bbb48d37745702ef97f7f6f', 'training.py': 'efe398febcd2485290fd6401a3c2aaa10991679c3cd8f152d73d308f9187c7f3', 'metrics.py': '39f5ec037ac66e59fd338ce5d4efe99abc71ed059294fca5e03a145ed0f9fd84', 'training_data.py': '6b16cc0d66276f782c9e839ec37fe6dbce703723f079ef706ef1be94a74cb684', 'reproducibility.py': 'ef59e1eb6453cfb149d11b6a4f2c7227d1aa8a8b7f354fb1a9a9b6e1590296aa', 'source_preparation/__init__.py': '0cb77d02f74e63498b4371acf27f7dfc94f1c69f70013d517b0a670d2f802d4b', 'source_preparation/archive.py': '127cb079e188fe4f1818ae1e7e183e9bef903df6e6be0972b2bfa3e64cd2c525', 'source_preparation/download.py': '898528f8ead6b7786801fb7bacb3075cd6417bd423f9ad64093ad39736f4db6f', 'source_preparation/duyu.py': 'b930e81175bf1981c5f250f5feac99424e576c8c77230c70cb64bace66bec4ed', 'source_preparation/aliases.py': 'cfab2e1f31bee366fedc27aaf4d81415cf89ff07f49ba8122ec0536a5b4da7cf', 'preparation/__init__.py': '81312216731c146a143da131f429ec1b420c038ebeb9150ae1cff482718c0b29', 'preparation/config.py': '5a7c15cf89693a4e8205e962c9994d665078564f6aebecad50add76d394d882f', 'preparation/embeddings.py': '2dfd1d57fd7610ba27fa6445299a505a776333765e408f305ce053272553fcd7'}
for relative in SOURCE_HASHES:
    (SDK_ROOT / "phonetics" / relative).parent.mkdir(parents=True, exist_ok=True)


### `__init__.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/__init__.py"
"""Mandarin phonetics public API with lazy optional ML dependencies."""
from importlib import import_module
_EXPORTS = {'Encoder': ('.model', 'SharedEncoder'), 'create_loaders': ('.data', 'create_loaders'), 'load_experiment': ('.experiment', 'load_experiment'), 'prepare_experiment': ('.experiment', 'prepare_experiment'), 'validate_experiment': ('.experiment', 'validate_experiment'), 'default_experiment_config': ('.experiment', 'default_experiment_config'), 'make_generator': ('.reproducibility', 'make_generator'), 'seed_everything': ('.reproducibility', 'seed_everything'), 'train_experiment': ('.training', 'train_experiment')}
_EXPORTS.update({"prepare_character_corpus": (".corpus", "prepare_character_corpus"),
                 "prepare_sound_inventory": (".sound_inventory", "prepare_sound_inventory"),
                 "prepare_acoustic_targets": (".acoustic", "prepare_acoustic_targets"),
                 "assemble_character_data": (".acoustic", "assemble_character_data"),
                 "select_seed": (".experiment", "select_seed")})
__all__=list(_EXPORTS)
def __getattr__(name):
    try:module,attribute=_EXPORTS[name]
    except KeyError:raise AttributeError(name) from None
    return getattr(import_module(module,__name__),attribute)


### `__main__.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/__main__.py"
"""Run the public command line interface with python -m phonetics."""
from .cli import main

raise SystemExit(main())


### `cli.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/cli.py"
"""Command-line entry point for reproducible data preparation and training."""
from __future__ import annotations

import argparse
import json
from pathlib import Path
import zipfile


def main(argv=None):
    parser = argparse.ArgumentParser(prog="phonetics")
    commands = parser.add_subparsers(dest="command", required=True)
    download = commands.add_parser("download-sounds", help="Download and verify the official Mandarin archive")
    download.add_argument("--output", required=True, help="Source bundle directory; existing content must match the source pin")
    download.add_argument("--timeout", type=float, default=60, help="Network read timeout in seconds")
    for stage in ("export-corpus", "prepare-inventory", "prepare-corpus", "prepare-embeddings",
                  "prepare-data", "prepare-experiment", "prepare"):
        command = commands.add_parser(stage, help=f"Run the {stage} stage using a pinned configuration")
        command.add_argument("--config", required=True, help="JSON configuration; paths resolve relative to this file")
        command.add_argument("--allow-inference", action="store_true", help="Explicitly enable acoustic model inference")
    train = commands.add_parser("train", help="Train one objective over the configured seeds")
    train.add_argument("--experiment", required=True)
    train.add_argument("--manifest-sha256", required=True)
    train.add_argument("--model", required=True)
    train.add_argument("--device", required=True)
    train.add_argument("--output", required=True)
    train.add_argument("--run-id")
    train.add_argument("--allow-training", action="store_true")
    args = parser.parse_args(argv)
    try:
        if args.command == "download-sounds":
            from .source_preparation.download import download_sounds, CHINESE_LESSONS_URL, CHINESE_LESSONS_SHA256
            path = download_sounds(args.output, timeout=args.timeout)
            print(json.dumps({"path": str(path), "url": CHINESE_LESSONS_URL,
                              "sha256": CHINESE_LESSONS_SHA256}, sort_keys=True))
        elif args.command == "train":
            if not args.allow_training:
                raise PermissionError("training requires --allow-training after reviewing inputs and device")
            from .training import train_experiment
            result = train_experiment(Path(args.experiment), model_id=args.model, device=args.device,
                                      runs_root=Path(args.output), run_id=args.run_id,
                                      expected_manifest_sha256=args.manifest_sha256)
            print(json.dumps({"path": str(result["run_dir"]), "status": result["manifest"]["status"]}))
        else:
            from .pipeline import load_config, run_stage
            path = run_stage(load_config(args.config), args.command, allow_inference=args.allow_inference)
            print(json.dumps({"path": str(path), "stage": args.command}, sort_keys=True))
    except (OSError, ValueError, RuntimeError, zipfile.BadZipFile) as exc:
        parser.exit(1, f"Error: {exc}\n")
    return 0


### `pipeline.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/pipeline.py"
"""Configuration-driven stages shared by the CLI and portable notebooks."""
from __future__ import annotations

import json
from pathlib import Path

from .artifacts import safe_artifact, sha256_file


def load_config(path):
    path = Path(path).absolute()
    value = json.loads(path.read_text(encoding="utf-8"))
    required = {"schema_version", "paths", "corpus", "model", "acoustic", "experiment_id", "experiment"}
    if not isinstance(value, dict) or set(value) != required or type(value["schema_version"]) is not int or value["schema_version"] != 1:
        raise ValueError("invalid pipeline configuration schema")
    paths = value["paths"]
    names = {"source", "export", "inventory", "corpus", "acoustic", "prepared", "experiments", "cache", "runs"}
    if not isinstance(paths, dict) or set(paths) != names:
        raise ValueError("pipeline paths must explicitly select all stage destinations")
    def resolve(raw):
        if not isinstance(raw, str) or not raw:
            raise ValueError("paths must be nonempty strings")
        p = Path(raw).expanduser()
        return str((path.parent / p).resolve()) if not p.is_absolute() else str(p.resolve())
    value["paths"] = {key: resolve(raw) for key, raw in paths.items()}
    if len(set(value["paths"].values())) != len(names):
        raise ValueError("stage output directories must be distinct")
    acoustic = value["acoustic"]
    if not isinstance(acoustic, dict) or set(acoustic) != {"existing_bundle", "manifest_sha256"}:
        raise ValueError("acoustic config requires existing_bundle and manifest_sha256, or both null")
    if (acoustic["existing_bundle"] is None) != (acoustic["manifest_sha256"] is None):
        raise ValueError("acoustic bundle and manifest pin must be selected together")
    if acoustic["existing_bundle"] is not None:
        acoustic["existing_bundle"] = resolve(acoustic["existing_bundle"])
    if not isinstance(value["experiment"], dict) or "sample_size" not in value["experiment"]:
        raise ValueError("explicit experiment.sample_size is required")
    return value


def export_corpus(config):
    from .source_preparation.duyu import export_duyu, _validate_args
    spec = config["corpus"]
    if set(spec) != {"revision", "split", "columns", "tone_convention"}:
        raise ValueError("invalid corpus source parameters")
    _validate_args(**spec)
    target = Path(config["paths"]["export"])
    if target.exists() or target.is_symlink():
        if target.is_symlink() or not target.is_dir():
            raise ValueError("corpus export must be a non-symlink directory")
        provenance = json.loads(safe_artifact(target, "provenance.json").read_text(encoding="utf-8"))
        if any(provenance.get(k) != v for k, v in spec.items()):
            raise ValueError("corpus export parameters changed; use a new output ID")
        required = {"corpus.csv", "source_rows.jsonl", "rejections.jsonl", "metrics.json"}
        if set(provenance.get("files", {})) != required:
            raise ValueError("corpus export is incomplete")
        for name, digest in provenance["files"].items():
            if sha256_file(safe_artifact(target, name)) != digest:
                raise ValueError("corpus export checksum mismatch")
        return target
    return export_duyu(target, **spec, allow_download=True)


def run_stage(config, stage, *, allow_inference=False):
    paths = config["paths"]
    if stage == "export-corpus":
        return export_corpus(config)
    if stage == "download-sounds":
        from .source_preparation.download import download_sounds
        return download_sounds(paths["source"])
    if stage == "prepare-inventory":
        from .sound_inventory import prepare_sound_inventory
        return prepare_sound_inventory(paths["source"], paths["inventory"])
    if stage == "prepare-corpus":
        from .corpus import prepare_character_corpus
        export = export_corpus(config)
        return prepare_character_corpus(export / "corpus.csv", paths["inventory"], paths["corpus"],
                                        expected_sha256=sha256_file(export / "corpus.csv"))
    if stage == "prepare-embeddings":
        spec = config["acoustic"]
        if spec["existing_bundle"] is None and not allow_inference:
            raise PermissionError("real acoustic inference requires --allow-inference")
        from .acoustic import prepare_acoustic_targets
        return prepare_acoustic_targets(paths["inventory"], paths["acoustic"], model=config["model"],
                                        cache_root=paths["cache"], existing_bundle=spec["existing_bundle"],
                                        existing_manifest_sha256=spec["manifest_sha256"])
    if stage == "prepare-data":
        from .acoustic import assemble_character_data
        return assemble_character_data(paths["corpus"], paths["acoustic"], paths["prepared"])
    if stage == "prepare-experiment":
        from .experiment import prepare_experiment
        return prepare_experiment(config["experiment_id"], prepared_root=paths["prepared"],
                                  config=config["experiment"], output_root=paths["experiments"])
    if stage == "prepare":
        if config["acoustic"]["existing_bundle"] is None and not allow_inference:
            raise PermissionError("preparation would require inference; select an existing bundle or --allow-inference")
        result = None
        for name in ("export-corpus", "download-sounds", "prepare-inventory", "prepare-corpus",
                     "prepare-embeddings", "prepare-data", "prepare-experiment"):
            result = run_stage(config, name, allow_inference=allow_inference)
        return result
    raise ValueError(f"unknown preparation stage: {stage}")


### `artifacts.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/artifacts.py"
"""Small, deterministic artifact helpers for prepared datasets."""

from __future__ import annotations

import hashlib
import json
from pathlib import Path


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: str | Path, value) -> None:
    Path(path).write_text(json.dumps(value, ensure_ascii=False, sort_keys=True,
                                     indent=2) + "\n", encoding="utf-8")


def write_jsonl(path: str | Path, rows) -> None:
    Path(path).write_text("".join(json.dumps(row, ensure_ascii=False, sort_keys=True)
                                  + "\n" for row in rows), encoding="utf-8")


def read_jsonl(path: str | Path) -> list[dict]:
    result = []
    for number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        try:
            value = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"invalid JSONL at line {number}") from exc
        if not isinstance(value, dict):
            raise ValueError(f"JSONL row {number} is not an object")
        result.append(value)
    return result


def safe_artifact(root: str | Path, relative: str) -> Path:
    root = Path(root).resolve()
    rel = Path(relative)
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"unsafe artifact path: {relative}")
    path = (root / rel).resolve()
    if path != root and root not in path.parents:
        raise ValueError(f"unsafe artifact path: {relative}")
    cursor = root
    for part in rel.parts:
        cursor = cursor / part
        if cursor.exists() and cursor.is_symlink():
            raise ValueError(f"symlink artifact path: {relative}")
    return path


### `bundles.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/bundles.py"
"""Content-addressed validation and directory publication for pipeline stages."""
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
import shutil
import tempfile
from contextlib import contextmanager

from .artifacts import safe_artifact, sha256_file, write_json


def content_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(",", ":"), allow_nan=False).encode("utf-8")).hexdigest()


def validate_bundle(path, kind, *, identity=None):
    root = Path(path).absolute()
    if root.is_symlink() or not root.is_dir():
        raise ValueError("bundle must be a non-symlink directory")
    manifest_path = safe_artifact(root, "manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != 1 or manifest.get("bundle_type") != kind:
        raise ValueError("unsupported bundle schema or type")
    if manifest.get("identity_sha256") != content_hash(manifest.get("identity")):
        raise ValueError("bundle identity checksum mismatch")
    if identity is not None and manifest["identity"] != identity:
        raise ValueError("bundle inputs or parameters changed; use a new output ID")
    artifacts = manifest.get("artifacts")
    if not isinstance(artifacts, dict) or not artifacts or "manifest.json" in artifacts:
        raise ValueError("invalid bundle artifacts")
    actual = set()
    for item in root.rglob("*"):
        if item.is_symlink():
            raise ValueError("bundle must not contain symlinks")
        if item.is_file() and item != manifest_path:
            actual.add(item.relative_to(root).as_posix())
    if actual != set(artifacts):
        raise ValueError("bundle artifact set mismatch")
    for name, digest in artifacts.items():
        if sha256_file(safe_artifact(root, name)) != digest:
            raise ValueError(f"bundle artifact checksum mismatch: {name}")
    return manifest


def seal_bundle(root, kind, identity):
    root = Path(root)
    artifacts = {p.relative_to(root).as_posix(): sha256_file(p)
                 for p in sorted(root.rglob("*")) if p.is_file() and p.name != "manifest.json"}
    write_json(root / "manifest.json", {"schema_version": 1, "bundle_type": kind,
               "identity": identity, "identity_sha256": content_hash(identity), "artifacts": artifacts})
    validate_bundle(root, kind, identity=identity)


@contextmanager
def staging_directory(destination):
    """Stage on the destination filesystem; never replace existing bundles."""
    target = Path(destination).absolute()
    if target.exists() or target.is_symlink():
        raise FileExistsError(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(tempfile.mkdtemp(prefix=f".{target.name}.", dir=target.parent))
    try:
        yield temporary
        if target.exists() or target.is_symlink():
            raise FileExistsError(target)
        os.rename(temporary, target)
    finally:
        if temporary.exists():
            shutil.rmtree(temporary)


### `data_contract.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/data_contract.py"
"""Character identity and acoustic target validation shared by the pipeline."""
import hashlib
import json


def sample_id(text, labels):
    return hashlib.sha256(json.dumps([text, labels], ensure_ascii=False,
                                    separators=(",", ":")).encode("utf-8")).hexdigest()


def is_han(character):
    code = ord(character)
    return (0x3400 <= code <= 0x4DBF or 0x4E00 <= code <= 0x9FFF
            or 0xF900 <= code <= 0xFAFF or 0x20000 <= code <= 0x323AF)


def validate_acoustic(mapping, *, expected_keys=None):
    import numpy as np

    if not isinstance(mapping, dict) or not mapping:
        raise ValueError("acoustic mapping must be a non-empty dict")
    if any(not isinstance(key, str) or not key for key in mapping):
        raise ValueError("invalid acoustic key")
    if expected_keys is not None and set(mapping) != set(expected_keys):
        raise ValueError("acoustic keys disagree with inventory")
    dimension = None
    for key, value in sorted(mapping.items()):
        vector = np.asarray(value, dtype=np.float32)
        if vector.ndim != 1 or not vector.size:
            raise ValueError(f"invalid acoustic vector: {key}")
        if dimension is None:
            dimension = vector.size
        if (vector.size != dimension or not np.isfinite(vector).all()
                or np.linalg.norm(vector.astype(np.float64)) <= 1e-12):
            raise ValueError(f"invalid acoustic vector: {key}")
    return dimension


### `pinyin.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/pinyin.py"
"""Project TONE3 input contract backed by pinned pypinyin inventories."""
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import version
import re

from pypinyin.constants import PINYIN_DICT
from pypinyin.contrib.tone_convert import to_finals, to_initials, to_normal

PINYIN_CONTRACT_VERSION = "pypinyin-tone3-1"


def _require_pypinyin_version():
    installed = version("pypinyin")
    if installed != "0.55.0":
        raise RuntimeError(f"pypinyin 0.55.0 required, found {installed}")
    return installed


_PYPINYIN_VERSION = _require_pypinyin_version()
# Dictionary readings contain tone marks; external project inputs do not.
VALID_SYLLABLES = frozenset(
    to_normal(reading, v_to_u=False).lower()
    for readings in PINYIN_DICT.values()
    for reading in readings.split(",")
)
_INPUT_SYLLABLES = tuple(sorted(base for base in VALID_SYLLABLES if re.fullmatch(r"[a-z]+", base)))
_STRICT_COMPONENTS = tuple(
    (to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False))
    for base in _INPUT_SYLLABLES
)
INITIALS = sorted({initial for initial, final in _STRICT_COMPONENTS if final})
FINALS = sorted({final for _, final in _STRICT_COMPONENTS if final})
TONES = [str(tone) for tone in range(1, 6)]
PARSER_PROVENANCE = {
    "library": "pypinyin", "version": _PYPINYIN_VERSION,
    "api": "pypinyin.contrib.tone_convert", "strict": True,
    "input_format": "lowercase-ascii-tone3", "umlaut_storage": "v",
    "input_alias": "u:", "missing_tone": "5", "empty_final": "reject",
    "inventory_source": "pypinyin.constants.PINYIN_DICT",
    "inventory_sha256": sha256("\n".join(sorted(VALID_SYLLABLES)).encode("utf-8")).hexdigest(),
    "vocabulary_sha256": sha256(repr((INITIALS, FINALS, TONES)).encode("utf-8")).hexdigest(),
}


@dataclass(frozen=True)
class PinyinComponents:
    initial: str
    final: str
    tone: str


def canonical_label(token: str, *, tone_convention="neutral_unmarked") -> str:
    """Accept one lowercase TONE3 syllable, allowing u: and unmarked neutral."""
    if tone_convention not in {"neutral_unmarked", "explicit"}:
        raise ValueError(f"unsupported tone convention: {tone_convention!r}")
    if not isinstance(token, str) or not token:
        raise ValueError(f"Pinyin must be a non-empty string: {token!r}")
    if re.fullmatch(r"(?:[a-z]|u:)+[1-5]?", token) is None:
        raise ValueError(f"invalid Pinyin format (lowercase ASCII TONE3 required): {token!r}")
    canonical = token.replace("u:", "v")
    if canonical[-1] not in TONES:
        if tone_convention == "explicit":
            raise ValueError(f"explicit tone required: {token!r}")
        canonical += "5"
    if re.fullmatch(r"[a-z]+[1-5]", canonical) is None:
        raise ValueError(f"invalid canonical Pinyin format: {token!r}")
    base = canonical[:-1]
    if base not in VALID_SYLLABLES:
        raise ValueError(f"syllable absent from pypinyin dictionary: {token!r}")
    initial = to_initials(base, strict=True)
    final = to_finals(base, strict=True, v_to_u=False)
    if not final or initial not in INITIALS or final not in FINALS:
        raise ValueError(f"unsupported strict Pinyin components (non-empty final required): {token!r}")
    return canonical


def validate_canonical_label(token: str) -> str:
    """Reject noncanonical stored labels instead of silently repairing them."""
    if canonical_label(token, tone_convention="explicit") != token:
        raise ValueError(f"stored Pinyin label is not canonical: {token!r}")
    return token


def parse_components(token: str, *, tone_convention="neutral_unmarked") -> PinyinComponents:
    canonical = canonical_label(token, tone_convention=tone_convention)
    base = canonical[:-1]
    return PinyinComponents(to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False), canonical[-1])


def encode_components(components: PinyinComponents) -> tuple[int, int, int]:
    try:
        return INITIALS.index(components.initial), FINALS.index(components.final), TONES.index(components.tone)
    except ValueError as exc:
        raise ValueError(f"components outside Pinyin vocabulary: {components!r}") from exc


def canonical_labels(tokens, *, tone_convention="neutral_unmarked"):
    return [canonical_label(token, tone_convention=tone_convention) for token in tokens]


def decompose(token: str, *, tone_convention="neutral_unmarked") -> tuple[int, int, int]:
    return encode_components(parse_components(token, tone_convention=tone_convention))


### `objectives.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/objectives.py"
"""Objective identifiers and arbitrary finite acoustic loss weights."""
from __future__ import annotations

import math
import re
from numbers import Real

BASE_MODEL_IDS = ("syllable", "components", "acoustic")
COMBINED_PREFIX = "components-acoustic-lambda-"


def combined_model_id(weight: Real) -> str:
    """Encode a finite coefficient without truncating its float precision."""
    if isinstance(weight, bool) or not isinstance(weight, Real):
        raise ValueError("lambda must be a finite real number")
    try:
        value = float(weight)
    except (OverflowError, ValueError) as exc:
        raise ValueError("lambda must be a finite real number") from exc
    if not math.isfinite(value):
        raise ValueError("lambda must be a finite real number")
    token = str(int(value)) if value.is_integer() and abs(value) < 1e16 else repr(value).replace("e+", "e")
    return COMBINED_PREFIX + token


def parse_model_id(model_id: str) -> tuple[str, float | None]:
    """Return the objective family and the combined acoustic coefficient."""
    if isinstance(model_id, str) and model_id in BASE_MODEL_IDS:
        return model_id, None
    if not isinstance(model_id, str) or not model_id.startswith(COMBINED_PREFIX):
        raise ValueError("unknown objective model_id")
    token = model_id[len(COMBINED_PREFIX):]
    if not re.fullmatch(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?", token):
        raise ValueError("combined model_id must contain a finite numeric lambda")
    weight = float(token)
    if not math.isfinite(weight):
        raise ValueError("lambda must be a finite real number")
    return "components-acoustic", weight


def canonical_model_id(model_id: str) -> str:
    family, weight = parse_model_id(model_id)
    return combined_model_id(weight) if family == "components-acoustic" else family


### `sound_inventory.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/sound_inventory.py"
"""Model-independent audio inventory built before acoustic inference."""
from __future__ import annotations

from pathlib import Path
import wave

from .artifacts import read_jsonl, sha256_file, write_jsonl
from .bundles import seal_bundle, staging_directory, validate_bundle
from .pinyin import PARSER_PROVENANCE, canonical_label, validate_canonical_label
from .source_preparation.archive import unpack_sounds
from .source_preparation.download import ARCHIVE_NAME, validate_source_bundle
from .source_preparation.aliases import CHINESE_LESSONS_ALIASES


def inspect_audio(path):
    """Read every decoded frame, rather than trusting only container headers."""
    path = Path(path)
    if path.suffix.lower() == ".wav":
        with wave.open(str(path), "rb") as handle:
            rate, channels, frames = handle.getframerate(), handle.getnchannels(), handle.getnframes()
            data = handle.readframes(frames)
            if len(data) != frames * channels * handle.getsampwidth():
                raise ValueError(f"truncated audio: {path.name}")
    else:
        try:
            import soundfile as sf
        except ImportError as exc:
            raise RuntimeError("Install the audio dependency group to decode MP3 inputs") from exc
        with sf.SoundFile(path) as handle:
            rate, channels, frames = handle.samplerate, handle.channels, len(handle)
            read_frames = sum(len(block) for block in handle.blocks(blocksize=65536))
            if read_frames != frames:
                raise ValueError(f"truncated audio: {path.name}")
    if rate <= 0 or channels <= 0 or frames <= 0:
        raise ValueError(f"empty or invalid audio: {path.name}")
    return {"sample_rate": rate, "channels": channels, "frames": frames}


def prepare_sound_inventory(source_bundle, destination):
    source = validate_source_bundle(source_bundle)
    identity = {"source_manifest_sha256": sha256_file(source / "manifest.json"),
                "parser": PARSER_PROVENANCE, "aliases": CHINESE_LESSONS_ALIASES,
                "tone_convention": "neutral_unmarked", "contract": "sound-inventory-1"}
    target = Path(destination).absolute()
    if target.exists() or target.is_symlink():
        validate_sound_inventory(target, identity=identity)
        return target
    with staging_directory(target) as temporary:
        unpack_sounds(source / ARCHIVE_NAME, temporary / "sounds",
                      expected_sha256=sha256_file(source / ARCHIVE_NAME))
        records, rejected = [], []
        seen = set()
        for path in sorted((temporary / "sounds").iterdir()):
            # Validate decoding even for filenames outside the Pinyin domain.
            info = inspect_audio(path)
            raw = CHINESE_LESSONS_ALIASES.get(path.name, path.stem)
            try:
                label = canonical_label(raw, tone_convention="neutral_unmarked")
            except ValueError as exc:
                rejected.append({"file": path.name, "reason": "unsupported_pinyin", "detail": str(exc)})
                continue
            if label in seen:
                raise ValueError(f"duplicate canonical sound label: {label}")
            seen.add(label)
            records.append({"label": label, "relative_path": f"sounds/{path.name}",
                            "sha256": sha256_file(path), **info})
        if not records:
            raise ValueError("sound inventory is empty")
        write_jsonl(temporary / "inventory.jsonl", sorted(records, key=lambda r: r["label"]))
        write_jsonl(temporary / "rejections.jsonl", rejected)
        seal_bundle(temporary, "sound_inventory", identity)
        validate_sound_inventory(temporary)
    return target


def validate_sound_inventory(path, *, identity=None):
    manifest = validate_bundle(path, "sound_inventory", identity=identity)
    rows = read_jsonl(Path(path) / "inventory.jsonl")
    labels = [validate_canonical_label(row["label"]) for row in rows]
    if not labels or labels != sorted(set(labels)):
        raise ValueError("sound labels must be nonempty, sorted and unique")
    from .artifacts import safe_artifact
    for row in rows:
        audio = safe_artifact(path, row["relative_path"])
        if sha256_file(audio) != row["sha256"]:
            raise ValueError("inventory audio checksum mismatch")
        if any(type(row[key]) is not int or row[key] <= 0 for key in ("sample_rate", "channels", "frames")):
            raise ValueError("invalid inventory audio dimensions")
    return manifest


### `corpus.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/corpus.py"
"""Prepare the complete eligible character corpus without IDS or embeddings."""
from __future__ import annotations

from collections import Counter, defaultdict
import csv
import json
from pathlib import Path

from .artifacts import read_jsonl, sha256_file, write_json, write_jsonl
from .bundles import seal_bundle, staging_directory, validate_bundle
from .data_contract import is_han, sample_id
from .pinyin import PARSER_PROVENANCE, canonical_labels, validate_canonical_label
from .sound_inventory import validate_sound_inventory


def prepare_character_corpus(csv_path, inventory_path, destination, *, expected_sha256,
                             tone_convention="explicit", source_id="duyu"):
    csv_path, inventory_path = Path(csv_path), Path(inventory_path)
    if tone_convention not in {"explicit", "neutral_unmarked"}:
        raise ValueError("unsupported tone convention")
    if sha256_file(csv_path) != expected_sha256:
        raise ValueError("corpus source checksum mismatch")
    validate_sound_inventory(inventory_path)
    inventory = {r["label"] for r in read_jsonl(inventory_path / "inventory.jsonl")}
    identity = {"corpus_sha256": expected_sha256, "source_id": source_id,
                "inventory_manifest_sha256": sha256_file(inventory_path / "manifest.json"),
                "parser": PARSER_PROVENANCE, "tone_convention": tone_convention,
                "conflict_policy": "exclude_all_variants", "contract": "character-corpus-1"}
    target = Path(destination).absolute()
    if target.exists() or target.is_symlink():
        validate_character_corpus(target, identity=identity)
        return target
    accepted, rejected, source_ids = [], [], set()
    with csv_path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle, strict=True)
        fields = reader.fieldnames
        if (not fields or len(set(fields)) != len(fields) or not {"text", "pinyin"}.issubset(fields)
                or set(fields) - {"text", "pinyin", "source_id"}):
            raise ValueError("corpus headers must be text,pinyin[,source_id]")
        for number, raw in enumerate(reader, 2):
            if None in raw or any(value is None for value in raw.values()):
                raise ValueError(f"malformed corpus row {number}")
            sid = f"{source_id}:{raw.get('source_id') or f'row-{number}'}"
            if sid in source_ids:
                raise ValueError(f"duplicate corpus source ID: {sid}")
            source_ids.add(sid)
            text, tokens = raw["text"], raw["pinyin"].split()
            row = {"text": text, "source_ids": [sid], "source_row": number}
            reason = None
            if not text or not tokens:
                reason = "nonempty"
            elif any(not is_han(c) for c in text):
                reason = "characters"
            elif len(text) != len(tokens):
                reason = "alignment"
            try:
                labels = canonical_labels(tokens, tone_convention=tone_convention)
            except (ValueError, TypeError):
                labels = tokens
                reason = reason or "pinyin"
            row["labels"] = labels
            if reason:
                rejected.append({**row, "reason": reason})
            else:
                accepted.append(row)
    # Pronunciation conflicts are excluded before sound-coverage filtering, so
    # losing one reading to coverage cannot silently resolve an ambiguity.
    groups = defaultdict(list)
    for row in accepted:
        groups[row["text"]].append(row)
    rows = []
    duplicate_count = 0
    for text, members in sorted(groups.items()):
        variants = {tuple(row["labels"]) for row in members}
        if len(variants) > 1:
            rejected.extend({**row, "reason": "conflicting_pronunciation"} for row in members)
            continue
        labels = members[0]["labels"]
        if not set(labels).issubset(inventory):
            rejected.extend({**row, "reason": "sound_coverage",
                             "missing_labels": sorted(set(labels) - inventory)} for row in members)
            continue
        duplicate_count += len(members) - 1
        rows.append({"text": text, "labels": labels,
                     "source_ids": sorted(s for row in members for s in row["source_ids"]),
                     "sample_id": sample_id(text, labels), "sentence_length": len(text),
                     "positions": list(range(len(text)))})
    rows.sort(key=lambda row: row["sample_id"])
    if not rows:
        raise ValueError("no eligible corpus sentences remain")
    with staging_directory(target) as temporary:
        write_jsonl(temporary / "corpus.jsonl", rows)
        write_jsonl(temporary / "rejections.jsonl", rejected)
        write_jsonl(temporary / "inventory.jsonl", read_jsonl(inventory_path / "inventory.jsonl"))
        write_json(temporary / "statistics.json", {"input": len(source_ids), "selected": len(rows),
                   "rejected": len(rejected), "duplicates": duplicate_count,
                   "reasons": dict(sorted(Counter(r["reason"] for r in rejected).items()))})
        seal_bundle(temporary, "character_corpus", identity)
        validate_character_corpus(temporary)
    return target


def validate_character_corpus(path, *, identity=None):
    manifest = validate_bundle(path, "character_corpus", identity=identity)
    validate_corpus_records(path)
    return manifest


def validate_corpus_records(path):
    """Validate corpus semantics, also when embedded in prepared data."""
    rows = read_jsonl(Path(path) / "corpus.jsonl")
    inventory = {r["label"] for r in read_jsonl(Path(path) / "inventory.jsonl")}
    ids, texts, sources = [], set(), set()
    for row in rows:
        text, labels, sids = row["text"], row["labels"], row["source_ids"]
        if (not text or any(not is_han(c) for c in text) or len(text) != len(labels)
                or row["sentence_length"] != len(text) or row["positions"] != list(range(len(text)))):
            raise ValueError("corpus alignment or character contract mismatch")
        for label in labels:
            validate_canonical_label(label)
        if not set(labels).issubset(inventory) or row["sample_id"] != sample_id(text, labels):
            raise ValueError("corpus labels or identity mismatch")
        if text in texts or not sids or sids != sorted(set(sids)) or sources.intersection(sids):
            raise ValueError("duplicated corpus text or source identity")
        texts.add(text)
        sources.update(sids)
        ids.append(row["sample_id"])
    if not ids or ids != sorted(set(ids)):
        raise ValueError("corpus sample IDs must be nonempty, sorted and unique")
    stats = json.loads((Path(path) / "statistics.json").read_text(encoding="utf-8"))
    rejections = read_jsonl(Path(path) / "rejections.jsonl")
    if (stats.get("selected") != len(rows) or stats.get("rejected") != len(rejections)
            or stats.get("input") != len(rows) + len(rejections) + stats.get("duplicates", -1)):
        raise ValueError("corpus population statistics mismatch")


### `acoustic.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/acoustic.py"
"""Inventory-bound acoustic targets and character prepared datasets."""
from __future__ import annotations

import json
from pathlib import Path
import pickle
import shutil

from .artifacts import read_jsonl, safe_artifact, sha256_file, write_json, write_jsonl
from .bundles import seal_bundle, staging_directory, validate_bundle
from .corpus import validate_character_corpus, validate_corpus_records
from .data_contract import validate_acoustic
from .pinyin import PARSER_PROVENANCE, PINYIN_CONTRACT_VERSION, validate_canonical_label
from .sound_inventory import validate_sound_inventory


def inventory_signature(rows):
    return sorted((row["label"], row["sha256"]) for row in rows)


def prepare_acoustic_targets(inventory_path, destination, *, model, cache_root=None,
                             existing_bundle=None, existing_manifest_sha256=None, backend=None):
    """Compute targets or import a pinned historical acoustic-only bundle.

    Import accepts only the exact inventory/audio identity. Historical corpus,
    IDS, sampling and prepared schemas are never consumed by this operation.
    """
    import numpy as np
    from .preparation.config import validate_model_config

    model = validate_model_config(model)
    if len(model["revision"]) != 40 or any(c not in "0123456789abcdef" for c in model["revision"]):
        raise ValueError("model revision must be an exact 40-character commit SHA")
    inventory_path = Path(inventory_path)
    validate_sound_inventory(inventory_path)
    rows = read_jsonl(inventory_path / "inventory.jsonl")
    identity = {"inventory_manifest_sha256": sha256_file(inventory_path / "manifest.json"),
                "model": model, "contract": "acoustic-targets-1", "backend": "fixture" if backend else "transformers"}
    if existing_bundle is not None:
        if backend is not None or existing_manifest_sha256 is None:
            raise ValueError("existing acoustic bundle requires an explicit pin and no inference backend")
        root = Path(existing_bundle)
        if sha256_file(root / "manifest.json") != existing_manifest_sha256:
            raise ValueError("acoustic input manifest checksum mismatch")
        old = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
        if old.get("bundle_type") != "acoustic_embeddings" or old.get("schema_version") != 1:
            raise ValueError("unsupported acoustic input schema")
        required = {"pinyin_embeddings.pkl", "sound_inventory.jsonl", "config.json", "reports/inference.json"}
        if not required.issubset(old.get("artifacts", {})):
            raise ValueError("acoustic input is missing required authenticated artifacts")
        if old.get("pinyin_contract") != PINYIN_CONTRACT_VERSION or old.get("parser_provenance") != PARSER_PROVENANCE:
            raise ValueError("acoustic input Pinyin provenance mismatch")
        for name, digest in old["artifacts"].items():
            if sha256_file(safe_artifact(root, name)) != digest:
                raise ValueError("acoustic input artifact checksum mismatch")
        old_rows = read_jsonl(root / "sound_inventory.jsonl")
        if old.get("inventory_sha256") != sha256_file(root / "sound_inventory.jsonl"):
            raise ValueError("acoustic input inventory checksum mismatch")
        old_labels = [validate_canonical_label(r["label"]) for r in old_rows]
        if len(set(old_labels)) != len(old_labels) or not old_labels:
            raise ValueError("acoustic input inventory labels invalid")
        cfg = json.loads((root / "config.json").read_text(encoding="utf-8"))
        inference = json.loads((root / "reports/inference.json").read_text(encoding="utf-8"))
        if (cfg.get("bundle_id") != old.get("bundle_id")
                or {**validate_model_config(cfg["model"]), "backend": inference.get("backend")} != old.get("model")
                or inference != old.get("inference") or inference.get("backend") != "transformers"
                or any(type(inference.get(k)) is not int or inference[k] < 0 for k in ("computed", "cached", "errors"))
                or inference["errors"] != 0 or inference["computed"] + inference["cached"] != len(old_rows)
                or type(old.get("count")) is not int or old["count"] != len(old_rows)
                or type(old.get("dimension")) is not int or old["dimension"] <= 0):
            raise ValueError("acoustic input inference/model metadata mismatch")
        if inventory_signature(old_rows) != inventory_signature(rows):
            raise ValueError("acoustic input inventory/audio mismatch")
        old_model = dict(old["model"])
        old_model.pop("backend", None)
        # Device is execution provenance, not a difference in target semantics.
        if {k: v for k, v in old_model.items() if k != "device"} != {k: v for k, v in model.items() if k != "device"}:
            raise ValueError("acoustic model parameters mismatch")
        identity.update({"source_manifest_sha256": existing_manifest_sha256, "backend": "imported"})
    target = Path(destination).absolute()
    if target.exists() or target.is_symlink():
        validate_acoustic_targets(target, identity=identity)
        return target
    if existing_bundle is not None:
        with (root / "pinyin_embeddings.pkl").open("rb") as handle:
            vectors = pickle.load(handle)
        report = {"source_manifest_sha256": existing_manifest_sha256, "computed": 0, "imported": len(vectors)}
    else:
        if cache_root is None:
            raise ValueError("cache_root is required for acoustic inference")
        from .preparation.embeddings import embed_inventory
        runtime_rows = [{**row, "path": str(safe_artifact(inventory_path, row["relative_path"]))} for row in rows]
        vectors, report = embed_inventory(runtime_rows, model, cache_root, backend)
    validate_acoustic(vectors, expected_keys=[r["label"] for r in rows])
    if existing_bundle is not None and validate_acoustic(vectors) != old["dimension"]:
        raise ValueError("acoustic input target dimension mismatch")
    with staging_directory(target) as temporary:
        labels = sorted(vectors)
        np.save(temporary / "targets.npy", np.stack([vectors[k] for k in labels]).astype(np.float32), allow_pickle=False)
        write_json(temporary / "labels.json", labels)
        write_jsonl(temporary / "inventory.jsonl", rows)
        write_json(temporary / "inference.json", report)
        seal_bundle(temporary, "acoustic_targets", identity)
        validate_acoustic_targets(temporary)
    return target


def validate_acoustic_targets(path, *, identity=None):
    root = Path(path)
    manifest = validate_bundle(root, "acoustic_targets", identity=identity)
    validate_target_records(root)
    return manifest


def validate_target_records(root):
    import numpy as np
    root = Path(root)
    labels = json.loads((root / "labels.json").read_text(encoding="utf-8"))
    if not labels or labels != sorted(set(labels)):
        raise ValueError("acoustic labels must be sorted and unique")
    for label in labels:
        validate_canonical_label(label)
    matrix = np.load(root / "targets.npy", allow_pickle=False)
    if matrix.ndim != 2 or matrix.shape[0] != len(labels) or matrix.dtype != np.float32:
        raise ValueError("invalid acoustic target matrix")
    validate_acoustic(dict(zip(labels, matrix)), expected_keys=[r["label"] for r in read_jsonl(root / "inventory.jsonl")])


def assemble_character_data(corpus_path, acoustic_path, destination):
    corpus_path, acoustic_path = Path(corpus_path), Path(acoustic_path)
    cm = validate_character_corpus(corpus_path)
    am = validate_acoustic_targets(acoustic_path)
    if cm["identity"]["inventory_manifest_sha256"] != am["identity"]["inventory_manifest_sha256"]:
        raise ValueError("corpus and acoustic inventory provenance mismatch")
    if inventory_signature(read_jsonl(corpus_path / "inventory.jsonl")) != inventory_signature(read_jsonl(acoustic_path / "inventory.jsonl")):
        raise ValueError("corpus and acoustic inventory mismatch")
    identity = {"corpus_manifest_sha256": sha256_file(corpus_path / "manifest.json"),
                "acoustic_manifest_sha256": sha256_file(acoustic_path / "manifest.json"),
                "contract": "character-prepared-1"}
    target = Path(destination).absolute()
    if target.exists() or target.is_symlink():
        validate_character_data(target, identity=identity)
        return target
    with staging_directory(target) as temporary:
        for name in ("corpus.jsonl", "statistics.json", "rejections.jsonl", "inventory.jsonl"):
            shutil.copyfile(corpus_path / name, temporary / name)
        for name in ("labels.json", "targets.npy", "inference.json"):
            shutil.copyfile(acoustic_path / name, temporary / name)
        write_json(temporary / "corpus_manifest.json", cm)
        write_json(temporary / "acoustic_manifest.json", am)
        seal_bundle(temporary, "character_prepared", identity)
        validate_character_data(temporary)
    return target


def validate_character_data(path, *, identity=None):
    root = Path(path)
    manifest = validate_bundle(root, "character_prepared", identity=identity)
    for name, field in (("corpus_manifest.json", "corpus_manifest_sha256"),
                        ("acoustic_manifest.json", "acoustic_manifest_sha256")):
        if sha256_file(root / name) != manifest["identity"][field]:
            raise ValueError("prepared provenance checksum mismatch")
    cm = json.loads((root / "corpus_manifest.json").read_text(encoding="utf-8"))
    am = json.loads((root / "acoustic_manifest.json").read_text(encoding="utf-8"))
    from .bundles import content_hash
    for upstream, kind, names in (
        (cm, "character_corpus", ("corpus.jsonl", "statistics.json", "rejections.jsonl", "inventory.jsonl")),
        (am, "acoustic_targets", ("targets.npy", "labels.json", "inference.json", "inventory.jsonl"))):
        if (upstream.get("bundle_type") != kind or upstream.get("schema_version") != 1
                or upstream.get("identity_sha256") != content_hash(upstream.get("identity"))
                or set(upstream.get("artifacts", {})) != set(names)):
            raise ValueError("prepared upstream contract mismatch")
        for name in names:
            if sha256_file(root / name) != upstream["artifacts"][name]:
                raise ValueError(f"prepared upstream artifact mismatch: {name}")
    if cm["identity"]["inventory_manifest_sha256"] != am["identity"]["inventory_manifest_sha256"]:
        raise ValueError("prepared upstream inventory provenance mismatch")
    validate_corpus_records(root)
    validate_target_records(root)
    return manifest


### `experiment.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/experiment.py"
"""Immutable character experiments with a shared test and per-seed training data."""
from __future__ import annotations

import copy
import json
from pathlib import Path
import random
import shutil

import numpy as np

from .acoustic import validate_character_data
from .artifacts import read_jsonl, sha256_file, write_json
from .bundles import content_hash, seal_bundle, staging_directory, validate_bundle
from .data_contract import sample_id, is_han
from .objectives import canonical_model_id
from .pinyin import INITIALS, FINALS, TONES, decompose, validate_canonical_label

SCHEMA_VERSION = 4
DEFAULT_MODEL_IDS = ['syllable', 'components', 'acoustic', 'components-acoustic-lambda-1', 'components-acoustic-lambda-10']
_TRAIN = {'optimizer': 'Adam', 'learning_rate': 0.001, 'weight_decay': 0.0, 'batch_size': 32,
          'epochs': 20, 'scheduler': None, 'early_stopping': False, 'gradient_clipping': None, 'cosine_epsilon': 1e-8}
_ARCH = {'embedding_dim': 64, 'sentence_bilstm_hidden_per_direction': 64, 'sentence_bilstm_layers': 1,
         'context_dim': 128, 'classification_hidden_dim': 64, 'classification_dropout': 0.5}


def default_experiment_config(acoustic_dim=1024):
    return {'schema_version': SCHEMA_VERSION, 'sample_size': 100_000, 'seed_count': 5,
            'test_seed': 0, 'test_fraction': 0.1, 'validation_fraction': 0.1,
            'architecture': {**_ARCH, 'acoustic_dim': acoustic_dim},
            'training': copy.deepcopy(_TRAIN), 'model_ids': list(DEFAULT_MODEL_IDS)}


def _validate_config(value, acoustic_dim):
    expected = default_experiment_config(acoustic_dim)
    if not isinstance(value, dict) or set(value) != set(expected):
        raise ValueError('invalid character experiment config keys')
    cfg = copy.deepcopy(value)
    for key in ('sample_size', 'seed_count'):
        if type(cfg[key]) is not int or cfg[key] <= 0:
            raise ValueError(f'{key} must be a positive integer')
    for key in ('schema_version', 'test_seed', 'test_fraction', 'validation_fraction', 'architecture', 'training'):
        if content_hash(cfg[key]) != content_hash(expected[key]):
            raise ValueError(f'{key} conflicts with the experiment protocol')
    ids = cfg['model_ids']
    if not isinstance(ids, list) or not ids or [canonical_model_id(x) for x in ids] != ids or len(set(ids)) != len(ids):
        raise ValueError('model_ids must be unique canonical identifiers')
    if cfg['sample_size'] // 10 < 1:
        raise ValueError('sample_size must allow nonempty test and validation')
    return cfg


def stream_seed(seed, stream):
    return int(content_hash({'seed': seed, 'stream': stream})[:16], 16)


def make_splits(ids, config):
    ids = sorted(ids)
    if len(set(ids)) != len(ids):
        raise ValueError('duplicate sample IDs')
    n, count = config['sample_size'], config['seed_count']
    test_size, validation_size = n // 10, n // 10
    if len(ids) < n + test_size:
        raise ValueError('eligible corpus is too small for sample_size plus common test')
    test = sorted(random.Random(stream_seed(config['test_seed'], 'test')).sample(ids, test_size))
    pool = sorted(set(ids) - set(test))
    seeds = {}
    for seed in range(count):
        selected = random.Random(stream_seed(seed, 'sample')).sample(pool, n)
        random.Random(stream_seed(seed, 'validation')).shuffle(selected)
        seeds[str(seed)] = {'train': sorted(selected[validation_size:]),
                            'validation': sorted(selected[:validation_size])}
    return {'test': test, 'seeds': seeds}


def character_vocabulary(rows, train_ids):
    tokens = ['[PAD]', '[UNK]'] + sorted({c for sid in train_ids for c in rows[sid]['text']})
    return {'tokens': tokens, 'stoi': {c: i for i, c in enumerate(tokens)}, 'pad_id': 0, 'unk_id': 1}


def unk_statistics(rows, ids, vocabulary):
    stoi = vocabulary['stoi']
    unknown = [sum(c not in stoi for c in rows[sid]['text']) for sid in ids]
    positions = sum(len(rows[sid]['text']) for sid in ids)
    return {'sentences': len(ids), 'positions': positions, 'unknown_positions': sum(unknown),
            'unknown_sentences': sum(n > 0 for n in unknown),
            'position_rate': sum(unknown) / positions, 'sentence_rate': sum(n > 0 for n in unknown) / len(ids)}


def prepare_experiment(experiment_id, *, prepared_root, config, output_root=None):
    from .source_preparation.archive import WINDOWS_DEVICES, WINDOWS_FORBIDDEN
    if (not isinstance(experiment_id, str) or not experiment_id or experiment_id in {'.', '..'}
            or any(c in WINDOWS_FORBIDDEN for c in experiment_id) or experiment_id.endswith(('.', ' '))
            or experiment_id.split('.')[0].casefold() in WINDOWS_DEVICES):
        raise ValueError('experiment_id must be a safe path component')
    source = Path(prepared_root)
    validate_character_data(source)
    matrix = np.load(source / 'targets.npy', allow_pickle=False)
    cfg = _validate_config(config, matrix.shape[1])
    rows = {r['sample_id']: r for r in read_jsonl(source / 'corpus.jsonl')}
    splits = make_splits(rows, cfg)
    identity = {'experiment_id': experiment_id, 'prepared_manifest_sha256': sha256_file(source / 'manifest.json'),
                'config': cfg, 'contract': 'character-experiment-4', 'training_seeds': list(range(cfg['seed_count']))}
    target = Path(output_root or source.parent.parent / 'experiments').absolute() / experiment_id
    if target.exists() or target.is_symlink():
        validate_experiment(target, identity=identity)
        return target
    with staging_directory(target) as temporary:
        for name in ('corpus.jsonl', 'labels.json', 'targets.npy'):
            shutil.copyfile(source / name, temporary / name)
        shutil.copyfile(source / 'manifest.json', temporary / 'prepared_manifest.json')
        write_json(temporary / 'config.json', cfg)
        write_json(temporary / 'splits.json', splits)
        seed_root = temporary / 'seeds'
        seed_root.mkdir()
        for key, split in splits['seeds'].items():
            root = seed_root / key
            root.mkdir()
            vocabulary = character_vocabulary(rows, split['train'])
            write_json(root / 'vocabulary.json', vocabulary)
            write_json(root / 'diagnostics.json', {name: unk_statistics(rows, ids, vocabulary)
                       for name, ids in (('validation', split['validation']), ('test', splits['test']))})
            write_json(root / 'input_hashes.json', {'train': content_hash(split['train']),
                       'validation': content_hash(split['validation']), 'test': content_hash(splits['test']),
                       'vocabulary': content_hash(vocabulary)})
        seal_bundle(temporary, 'character_experiment', identity)
        validate_experiment(temporary)
    return target


def validate_experiment(path, *, identity=None):
    root = Path(path)
    manifest = validate_bundle(root, 'character_experiment', identity=identity)
    matrix = np.load(root / 'targets.npy', allow_pickle=False)
    if matrix.ndim != 2 or not matrix.shape[1]:
        raise ValueError('invalid acoustic target matrix shape')
    cfg = _validate_config(json.loads((root / 'config.json').read_text(encoding='utf-8')), matrix.shape[1])
    if cfg != manifest['identity']['config']:
        raise ValueError('experiment config identity mismatch')
    if manifest['identity'].get('training_seeds') != list(range(cfg['seed_count'])):
        raise ValueError('experiment seed identities mismatch')
    prepared = json.loads((root / 'prepared_manifest.json').read_text(encoding='utf-8'))
    if sha256_file(root / 'prepared_manifest.json') != manifest['identity']['prepared_manifest_sha256']:
        raise ValueError('prepared provenance pin mismatch')
    for name in ('corpus.jsonl', 'labels.json', 'targets.npy'):
        if prepared['artifacts'].get(name) != sha256_file(root / name):
            raise ValueError('experiment copied source mismatch')
    records = read_jsonl(root / 'corpus.jsonl')
    rows = {r['sample_id']: r for r in records}
    if len(rows) != len(records):
        raise ValueError('duplicate corpus sample IDs')
    labels = json.loads((root / 'labels.json').read_text(encoding='utf-8'))
    from .data_contract import validate_acoustic
    validate_acoustic(dict(zip(labels, matrix)), expected_keys=labels)
    if labels != sorted(set(labels)) or matrix.shape[0] != len(labels) or matrix.dtype != np.float32:
        raise ValueError('invalid target inventory')
    for row in records:
        if (not row['text'] or any(not is_han(c) for c in row['text'])
                or len(row['text']) != len(row['labels']) or row['sentence_length'] != len(row['text'])
                or row['positions'] != list(range(len(row['text'])))
                or not set(row['labels']).issubset(labels)
                or row['sample_id'] != sample_id(row['text'], row['labels'])):
            raise ValueError('sample identity mismatch')
        for label in row['labels']:
            validate_canonical_label(label)
    splits = json.loads((root / 'splits.json').read_text(encoding='utf-8'))
    if splits != make_splits(rows, cfg):
        raise ValueError('seed split reconstruction mismatch')
    for key, split in splits['seeds'].items():
        folder = root / 'seeds' / key
        vocabulary = character_vocabulary(rows, split['train'])
        if json.loads((folder / 'vocabulary.json').read_text(encoding='utf-8')) != vocabulary:
            raise ValueError('vocabulary must be built exclusively from this seed train')
        diagnostics = {name: unk_statistics(rows, ids, vocabulary) for name, ids in
                       (('validation', split['validation']), ('test', splits['test']))}
        if json.loads((folder / 'diagnostics.json').read_text(encoding='utf-8')) != diagnostics:
            raise ValueError('UNK diagnostics mismatch')
        hashes = {'train': content_hash(split['train']), 'validation': content_hash(split['validation']),
                  'test': content_hash(splits['test']), 'vocabulary': content_hash(vocabulary)}
        if json.loads((folder / 'input_hashes.json').read_text(encoding='utf-8')) != hashes:
            raise ValueError('seed input hash mismatch')
    return manifest


def load_experiment(path):
    root = Path(path)
    manifest = validate_experiment(root)
    return {'root': root, 'manifest': manifest, 'config': manifest['identity']['config'],
            'samples': {r['sample_id']: r for r in read_jsonl(root / 'corpus.jsonl')},
            'splits': json.loads((root / 'splits.json').read_text(encoding='utf-8')),
            'acoustic_targets': np.load(root / 'targets.npy', allow_pickle=False),
            'labels': json.loads((root / 'labels.json').read_text(encoding='utf-8'))}


def select_seed(experiment, seed):
    if type(seed) is not int or str(seed) not in experiment['splits']['seeds']:
        raise ValueError('seed is not present in experiment')
    vocabulary = json.loads((experiment['root'] / 'seeds' / str(seed) / 'vocabulary.json').read_text(encoding='utf-8'))
    labels = experiment['labels']
    label_ids = {label: i for i, label in enumerate(labels)}
    split = experiment['splits']['seeds'][str(seed)]
    samples = {}
    for sid in set(split['train'] + split['validation'] + experiment['splits']['test']):
        row = experiment['samples'][sid]
        samples[sid] = {**row, 'token_ids': [vocabulary['stoi'].get(c, 1) for c in row['text']],
                        'component_labels': [list(decompose(label)) for label in row['labels']],
                        'syllable_ids': [label_ids[label] for label in row['labels']],
                        'acoustic_ids': [label_ids[label] for label in row['labels']]}
    return {**experiment, 'seed': seed, 'samples': samples,
            'splits': {**split, 'test': experiment['splits']['test']},
            'vocab': {'characters': vocabulary, 'syllables': labels,
                      'labels': {'initials': INITIALS, 'finals': FINALS, 'tones': TONES}}}


### `model.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/model.py"
"""Shared character and sentence encoders with condition-specific readout heads."""
from __future__ import annotations

import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


from .objectives import canonical_model_id, parse_model_id


class SharedEncoder(nn.Module):
    """Embed characters and encode complete variable-length sentences."""
    context_dim = 128

    def __init__(self, char_vocab_size: int):
        super().__init__()
        self.embedding = nn.Embedding(char_vocab_size, 64, padding_idx=0)
        self.sentence = nn.LSTM(64, 64, num_layers=1, bidirectional=True, batch_first=True)

    def encode(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        token_ids = batch["token_ids"]
        lengths = batch["sentence_lengths"].to(dtype=torch.long)
        mask = batch["char_mask"].to(dtype=torch.bool)
        if token_ids.ndim != 2 or mask.shape != token_ids.shape:
            raise ValueError("invalid character batch shapes")
        if not torch.equal(mask, token_ids != 0):
            raise ValueError("character mask disagrees with padding")
        if not bool((lengths > 0).all()) or not torch.equal(lengths.cpu(), mask.sum(1).cpu()):
            raise ValueError("sentence lengths disagree with character mask")
        positions = torch.arange(token_ids.shape[1], device=token_ids.device)[None, :]
        if not torch.equal(mask, positions < lengths.to(token_ids.device)[:, None]):
            raise ValueError("character sequences must be right padded")
        packed = pack_padded_sequence(self.embedding(token_ids), lengths.cpu(), batch_first=True, enforce_sorted=False)
        output, _ = self.sentence(packed)
        hidden, _ = pad_packed_sequence(output, batch_first=True, total_length=token_ids.shape[1])
        return hidden, mask

    def forward(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encode(batch)


class ClassifierHead(nn.Module):
    def __init__(self, output_dim: int):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, output_dim))

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        return self.layers(hidden)


class TrainingModel(nn.Module):
    def __init__(self, encoder: SharedEncoder, model_id: str, vocab: dict, acoustic_dim: int):
        super().__init__()
        model_id = canonical_model_id(model_id)
        family, weight = parse_model_id(model_id)
        self.encoder = encoder
        self.model_id = model_id
        self.lambda_cosine = weight
        self.context_dim = 128
        labels = vocab["labels"]
        self.component_heads = nn.ModuleDict()
        if family in {"components", "components-acoustic"}:
            for name in ("initials", "finals", "tones"):
                self.component_heads[name] = ClassifierHead(len(labels[name]))
        self.syllable_head = ClassifierHead(len(vocab["syllables"])) if family == "syllable" else None
        self.acoustic_head = nn.Linear(128, int(acoustic_dim)) if family in {"acoustic", "components-acoustic"} else None

    def encode(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encoder.encode(batch)

    def forward(self, batch: dict) -> dict:
        hidden, mask = self.encoder.encode(batch)
        result = {"hidden": hidden, "mask": mask}
        if self.component_heads:
            result["component_logits"] = {name: head(hidden) for name, head in self.component_heads.items()}
        if self.syllable_head is not None:
            result["syllable_logits"] = self.syllable_head(hidden)
        if self.acoustic_head is not None:
            result["acoustic_prediction"] = self.acoustic_head(hidden)
        return result


def _construct_with_streams(config: dict, vocab: dict, model_id: str, seed: int) -> TrainingModel:
    """Initialize shared weights and heads from independent seed streams."""
    model_id = canonical_model_id(model_id)
    devices = []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(int(seed))
        encoder = SharedEncoder(len(vocab["characters"]["tokens"]))
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(int(seed) + 1_000_003)
        model = TrainingModel(encoder, model_id, vocab, int(config["architecture"]["acoustic_dim"]))
    return model


def build_encoder(config: dict, vocab: dict, model_id: str | None = None, seed: int = 0) -> TrainingModel:
    """Build a full model; the returned `.encode(batch)` is the shared export API."""
    architecture = config.get("architecture", {})
    if architecture.get("embedding_dim") != 64 or architecture.get("context_dim") != 128 or architecture.get("acoustic_dim", 0) <= 0:
        raise ValueError("experiment architecture violates encoder dimension contract")
    selected = model_id or config.get("model_id")
    if selected is None:
        raise ValueError("model_id is required to construct objective heads")
    return _construct_with_streams(config, vocab, selected, seed)


def shared_encoder_sha256(model: TrainingModel) -> str:
    import hashlib
    digest = hashlib.sha256()
    for name, tensor in sorted(model.encoder.state_dict().items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode("utf-8")); digest.update(str(value.dtype).encode("ascii"))
        digest.update(repr(tuple(value.shape)).encode("ascii")); digest.update(value.numpy().tobytes())
    return digest.hexdigest()


### `data.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/data.py"
"""Torch dataset and collation for seed-specific character experiment bundles."""
from __future__ import annotations

import torch
from torch.utils.data import DataLoader, Dataset

from .reproducibility import make_generator


class ExperimentDataset(Dataset):
    def __init__(self, sample_ids, samples):
        self.sample_ids = list(sample_ids)
        self.samples = samples

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, index):
        return self.samples[self.sample_ids[index]]


def collate_experiment(samples):
    if not samples:
        raise ValueError("cannot collate an empty batch")
    batch_size = len(samples)
    max_sentence = max(sample["sentence_length"] for sample in samples)
    token_ids = torch.zeros((batch_size, max_sentence), dtype=torch.long)
    component_labels = torch.full((batch_size, max_sentence, 3), -100, dtype=torch.long)
    syllable_ids = torch.full((batch_size, max_sentence), -100, dtype=torch.long)
    acoustic_ids = torch.full((batch_size, max_sentence), -1, dtype=torch.long)
    sentence_lengths = torch.empty(batch_size, dtype=torch.long)
    for row_index, sample in enumerate(samples):
        n = sample["sentence_length"]
        sentence_lengths[row_index] = n
        sequence = sample["token_ids"]
        if len(sequence) != n or any(type(token) is not int or token <= 0 for token in sequence):
            raise ValueError("invalid character token sequence")
        token_ids[row_index, :n] = torch.as_tensor(sequence, dtype=torch.long)
        component_labels[row_index, :n] = torch.as_tensor(sample["component_labels"], dtype=torch.long)
        syllable_ids[row_index, :n] = torch.as_tensor(sample["syllable_ids"], dtype=torch.long)
        acoustic_ids[row_index, :n] = torch.as_tensor(sample["acoustic_ids"], dtype=torch.long)
    char_mask = token_ids > 0
    return {
        "token_ids": token_ids,
        "sentence_lengths": sentence_lengths, "char_mask": char_mask,
        "component_labels": component_labels, "syllable_ids": syllable_ids,
        "acoustic_ids": acoustic_ids,
        "sample_ids": [sample["sample_id"] for sample in samples],
        "texts": [sample["text"] for sample in samples],
        "positions": [sample["positions"] for sample in samples],
    }


def create_loaders(experiment: dict, seed: int, batch_size: int | None = None, num_workers: int = 0):
    manifest = experiment.get("manifest", {})
    if manifest.get("bundle_type") != "character_experiment" or experiment.get("seed") != seed:
        raise ValueError("training loaders require a validated seed-specific character experiment")
    if type(seed) is not int or seed < 0:
        raise ValueError("loader seed must be a nonnegative integer")
    batch = batch_size or experiment["config"].get("training", {}).get("batch_size", 32)
    if type(batch) is not int or batch <= 0 or type(num_workers) is not int or num_workers < 0:
        raise ValueError("batch_size must be positive and num_workers nonnegative")
    result = {}
    for stream, split in enumerate(("train", "validation", "test"), start=1):
        dataset = ExperimentDataset(experiment["splits"][split], experiment["samples"])
        result[split] = DataLoader(dataset, batch_size=batch, shuffle=split == "train",
                                   generator=make_generator(seed, stream), num_workers=num_workers,
                                   collate_fn=collate_experiment)
    return result


### `training.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/training.py"
"""Sequential seed-specific character training runs for the selected objective."""
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import shutil
import statistics
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

from .artifacts import sha256_file, write_json
from .data import create_loaders
from .experiment import load_experiment, select_seed
from .metrics import batch_objective, finalize_epoch
from .model import build_encoder, shared_encoder_sha256
from .objectives import canonical_model_id, parse_model_id
from .reproducibility import rng_snapshot, seed_everything

CODE_FILES = ("training.py", "model.py", "data.py", "metrics.py", "reproducibility.py", "experiment.py",
              "training_data.py", "objectives.py", "pinyin.py", "artifacts.py", "data_contract.py", "bundles.py", "acoustic.py", "corpus.py", "sound_inventory.py")


class TrainingStepFailure(FloatingPointError):
    """A non-finite training step with JSON-safe diagnostic context."""

    def __init__(self, message, diagnostic):
        super().__init__(message)
        self.diagnostic = diagnostic


def _nonfinite_status(value):
    if math.isnan(value): return "nan"
    return "positive_infinity" if value > 0 else "negative_infinity"


def _gradient_summary(norm_values, nonfinite_count):
    return {"mean": statistics.fmean(norm_values) if norm_values else 0.0,
            "std": statistics.pstdev(norm_values) if len(norm_values) > 1 else 0.0,
            "max": max(norm_values) if norm_values else 0.0,
            "nonfinite_count": nonfinite_count}


def _failure_diagnostic(event, gradient_norms, nonfinite_count):
    """Retain successful work in the current epoch when a later step fails."""
    return {"nonfinite_count": int(nonfinite_count),
            "gradient_norms": list(gradient_norms), "event": event}


def _jsonable_metrics(values):
    return {key: (float(value.detach().cpu()) if torch.is_tensor(value) else value) for key, value in values.items()}


def _move(batch, device):
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}


def _gradient_norm(model):
    squared_norm = torch.zeros((), dtype=torch.float64, device="cpu")
    finite = True
    for parameter in model.parameters():
        if parameter.grad is None:
            continue
        grad = parameter.grad.detach()
        if not bool(torch.isfinite(grad).all()):
            finite = False
        squared_norm += grad.to(device="cpu", dtype=torch.float64).square().sum()
    value = float(torch.sqrt(squared_norm).item())
    return value, finite and math.isfinite(value)


def _epoch(model, loader, acoustic_targets, model_id, device, *, optimizer=None, event_writer=None,
           epoch_number=None, step_start=0):
    training = optimizer is not None
    model.train(training)
    totals = {}
    gradient_norms = []
    nonfinite_gradients = 0
    step = step_start
    for batch in loader:
        batch = _move(batch, device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            outputs = model(batch)
            stats = batch_objective(outputs, batch, acoustic_targets, model_id)
            objective = stats["objective"]
            if not bool(torch.isfinite(objective)):
                value = float(objective.detach().cpu())
                record = {"event": "nonfinite_loss", "epoch": epoch_number, "step": step,
                          "loss": None, "loss_status": _nonfinite_status(value)}
                if event_writer: event_writer(record)
                raise TrainingStepFailure(f"non-finite loss at epoch {epoch_number}, step {step}",
                                          _failure_diagnostic(record, gradient_norms, nonfinite_gradients))
            if training:
                objective.backward()
                grad_norm, finite_grad = _gradient_norm(model)
                if not finite_grad:
                    nonfinite_gradients += 1
                    record = {"event": "nonfinite_gradient", "epoch": epoch_number, "step": step,
                              "gradient_norm": None, "gradient_norm_status": _nonfinite_status(grad_norm)}
                    if event_writer: event_writer(record)
                    raise TrainingStepFailure(f"non-finite gradient at epoch {epoch_number}, step {step}",
                                              _failure_diagnostic(record, gradient_norms, nonfinite_gradients))
                gradient_norms.append(grad_norm)
                optimizer.step()
            else:
                grad_norm = None
        detached = _jsonable_metrics(stats)
        for key, value in detached.items():
            if key == "objective":
                continue
            totals[key] = totals.get(key, 0) + value
        if event_writer:
            event_writer({"event": "step", "epoch": epoch_number, "step": step,
                          "losses_per_character": {key.removesuffix("_sum"): value / stats["characters"]
                                                     for key, value in detached.items() if key.endswith("_sum")},
                          "objective": detached["objective"], "gradient_norm": grad_norm})
        step += 1
    result = finalize_epoch(totals)
    result["steps"] = step - step_start
    result["gradient_norms"] = gradient_norms
    result["gradient_nonfinite_count"] = nonfinite_gradients
    return result, step


def _state_hash(state):
    digest = hashlib.sha256()
    for name, tensor in sorted(state.items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode()); digest.update(str(value.dtype).encode()); digest.update(repr(tuple(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def _test_summary(seed_results):
    metrics = {}
    completed = [row for row in seed_results if "test_metrics" in row]
    keys = sorted(set.intersection(*(set(row["test_metrics"]) for row in completed))) if completed else []
    for key in keys:
        values = [float(row["test_metrics"][key]) for row in completed]
        metrics[key] = {"mean": statistics.fmean(values), "std": statistics.pstdev(values)}
    return metrics


def _write_run_manifest(run_dir, manifest):
    files = {}
    for file in sorted(run_dir.rglob("*")):
        if file.is_file() and file.name != "manifest.json":
            files[file.relative_to(run_dir).as_posix()] = sha256_file(file)
    manifest["artifacts"] = files
    write_json(run_dir / "manifest.json", manifest)


def train_experiment(experiment_path, *, model_id: str, runs_root=None, device: str | None = None,
                     run_id: str | None = None, expected_manifest_sha256: str | None = None):
    """Train the selected model over configured seeds, sequentially, without resume."""
    model_id = canonical_model_id(model_id)
    _, lambda_cosine = parse_model_id(model_id)
    experiment = load_experiment(experiment_path)
    experiment_root = Path(experiment["root"])
    if expected_manifest_sha256 is not None:
        actual_manifest_sha256 = sha256_file(experiment_root / "manifest.json")
        if actual_manifest_sha256 != expected_manifest_sha256:
            raise ValueError("experiment manifest does not match the explicitly selected input pin")
    config = experiment["config"]
    if model_id not in config["model_ids"]:
        raise ValueError("selected model_id is not configured in the experiment")
    seeds = experiment["manifest"]["identity"]["training_seeds"]
    requested_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    target_device = torch.device(requested_device)
    if target_device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(f"selected device {requested_device!r} requires CUDA")
    experiment_id = experiment["manifest"]["identity"]["experiment_id"]
    run_id = run_id or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ-") + uuid.uuid4().hex[:8]
    if not isinstance(run_id, str) or not run_id or Path(run_id).name != run_id or "/" in run_id or "\\" in run_id:
        raise ValueError("run_id must be a safe path component")
    default_root = Path(__file__).resolve().parents[2] / "runs"
    output_root = Path(runs_root).resolve() if runs_root else default_root
    run_dir = output_root / experiment_id / model_id / run_id
    if run_dir.exists():
        raise FileExistsError(f"run already exists: {run_dir}")
    run_dir.parent.mkdir(parents=True, exist_ok=True)
    run_dir.mkdir(parents=False, exist_ok=False)
    manifest = {
        "schema_version": 1, "run_type": "training_run", "status": "running", "run_id": run_id,
        "experiment_id": experiment_id, "model_id": model_id, "lambda_cosine": lambda_cosine, "device": str(target_device), "seeds": seeds,
        "selection": "minimum per-seed validation objective; strict improvement keeps earliest tie",
        "experiment_hashes": {name: sha256_file(experiment_root / name) for name in
                               ("manifest.json", "config.json", "labels.json", "corpus.jsonl", "splits.json", "targets.npy")},
        "config": config, "code_sha256": {name: sha256_file(Path(__file__).with_name(name)) for name in CODE_FILES},
        "runtime": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
                    "cuda_available": torch.cuda.is_available(), "cuda_version": torch.version.cuda},
        "seed_results": [], "summary": None, "artifacts": {},
    }
    write_json(run_dir / "manifest.json", manifest)
    seed_results = []
    statuses = {str(seed): "pending" for seed in seeds}
    global_step = 0
    active_seed = None
    try:
        for seed in seeds:
            active_seed = seed
            norm_values = []
            nonfinite_count = 0
            seed_dir = run_dir / f"seed-{seed}"
            seed_dir.mkdir()
            statuses[str(seed)] = "running"
            event_path = seed_dir / "events.jsonl"
            def log_event(event):
                with event_path.open("a", encoding="utf-8", newline="\n") as handle:
                    handle.write(json.dumps(event, sort_keys=True, allow_nan=False) + "\n")
            seed_input = select_seed(experiment, seed)
            seed_inputs = json.loads((experiment_root / "seeds" / str(seed) / "input_hashes.json").read_text(encoding="utf-8"))
            seed_everything(seed)
            model = build_encoder(config, seed_input["vocab"], model_id=model_id, seed=seed).to(target_device)
            initial_state = {key: value.detach().cpu().clone() for key, value in model.encoder.state_dict().items()}
            shared_hash = _state_hash(initial_state)
            torch.save({"state_dict": initial_state, "seed": seed, "sha256": shared_hash,
                        "seed_input_hashes": seed_inputs, "experiment_hashes": manifest["experiment_hashes"]},
                       seed_dir / "shared_encoder.pt")
            loaders = create_loaders(seed_input, seed, batch_size=config["training"]["batch_size"], num_workers=0)
            acoustic = torch.as_tensor(experiment["acoustic_targets"], dtype=torch.float32, device=target_device)
            optimizer = torch.optim.Adam(model.parameters(), lr=config["training"]["learning_rate"], weight_decay=0.0)
            best = math.inf
            best_epoch = None
            best_validation = None
            epoch_history = []
            for epoch_num in range(1, config["training"]["epochs"] + 1):
                train_metrics, global_step = _epoch(model, loaders["train"], acoustic, model_id, target_device,
                                                    optimizer=optimizer, event_writer=log_event,
                                                    epoch_number=epoch_num, step_start=global_step)
                norm_values.extend(train_metrics.pop("gradient_norms"))
                nonfinite_count += train_metrics.pop("gradient_nonfinite_count")
                validation_metrics, _ = _epoch(model, loaders["validation"], acoustic, model_id, target_device,
                                               event_writer=log_event, epoch_number=epoch_num)
                val_objective = float(validation_metrics["objective"])
                epoch_history.append({"epoch": epoch_num, "train": train_metrics, "validation": validation_metrics})
                if val_objective < best:
                    best, best_epoch, best_validation = val_objective, epoch_num, validation_metrics
                    torch.save({"state_dict": model.state_dict(), "model_id": model_id, "lambda_cosine": lambda_cosine, "seed": seed,
                                "best_epoch": best_epoch, "validation_objective": best,
                                "shared_encoder_sha256": shared_hash,
                                "experiment_hashes": manifest["experiment_hashes"],
                                "config_sha256": sha256_file(experiment_root / "config.json"),
                                "vocab_sha256": sha256_file(experiment_root / "seeds" / str(seed) / "vocabulary.json"),
                                "seed_input_hashes": seed_inputs}, seed_dir / "best.pt")
                torch.save({"state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                            "seed": seed, "epoch": epoch_num, "rng_state": rng_snapshot(loaders["train"].generator),
                            "model_id": model_id, "seed_input_hashes": seed_inputs,
                            "experiment_hashes": manifest["experiment_hashes"],
                            "shared_encoder_sha256": shared_hash},
                           seed_dir / "last.pt")
                write_json(seed_dir / "history.json", {"epochs": epoch_history})
                log_event({"event": "epoch", "epoch": epoch_num, "train": train_metrics, "validation": validation_metrics,
                           "best_epoch": best_epoch, "best_validation_objective": best})
            best_checkpoint = torch.load(seed_dir / "best.pt", map_location=target_device, weights_only=True)
            model.load_state_dict(best_checkpoint["state_dict"], strict=True)
            test_metrics, _ = _epoch(model, loaders["test"], acoustic, model_id, target_device,
                                     event_writer=log_event, epoch_number="test")
            test_metrics.pop("gradient_norms"); test_metrics.pop("gradient_nonfinite_count")
            grad_summary = _gradient_summary(norm_values, nonfinite_count)
            result = {"seed": seed, "status": "complete", "best_epoch": best_epoch,
                      "best_validation_objective": best, "best_validation_metrics": best_validation,
                      "test_metrics": test_metrics, "gradient_norm": grad_summary,
                      "shared_encoder_sha256": shared_hash, "seed_input_hashes": seed_inputs}
            seed_results.append(result)
            statuses[str(seed)] = "complete"
            write_json(seed_dir / "summary.json", result)
            manifest["seed_results"] = seed_results
            manifest["seed_status"] = statuses
            write_json(run_dir / "summary.json", {"status": "running", "completed_seeds": len(seed_results),
                        "seed_results": seed_results, "test_metrics": _test_summary(seed_results)})
            _write_run_manifest(run_dir, manifest)
            active_seed = None
        gradient = [row["gradient_norm"] for row in seed_results]
        run_summary = {"seeds_completed": len(seed_results), "test_metrics": _test_summary(seed_results),
                       "gradient_norm": {"mean_of_seed_means": statistics.fmean(x["mean"] for x in gradient),
                                         "max_of_seed_maxima": max(x["max"] for x in gradient),
                                         "nonfinite_count": sum(x["nonfinite_count"] for x in gradient)}}
        manifest.update({"status": "complete", "seed_status": statuses, "seed_results": seed_results, "summary": run_summary})
        write_json(run_dir / "summary.json", {"status": "complete", **run_summary, "seed_results": seed_results})
        _write_run_manifest(run_dir, manifest)
        return {"run_dir": run_dir, "manifest": manifest, "summary": run_summary}
    except Exception as exc:
        failure_diagnostic = getattr(exc, "diagnostic", None)
        if active_seed is not None and statuses.get(str(active_seed)) == "running":
            prior_norms = list(locals().get("norm_values", []))
            prior_nonfinite = int(locals().get("nonfinite_count", 0))
            if failure_diagnostic:
                prior_norms.extend(failure_diagnostic.get("gradient_norms", []))
                prior_nonfinite += int(failure_diagnostic.get("nonfinite_count", 0))
            failed_result = {"seed": active_seed, "status": "failed",
                             "gradient_norm": _gradient_summary(prior_norms, prior_nonfinite)}
            if failure_diagnostic:
                failed_result["failure_event"] = failure_diagnostic["event"]
            seed_results.append(failed_result)
            statuses[str(active_seed)] = "failed"
            failed_dir = run_dir / f"seed-{active_seed}"
            if failed_dir.is_dir():
                write_json(failed_dir / "summary.json", failed_result)
        statuses.update({key: ("failed" if value == "running" else value) for key, value in statuses.items()})
        status = "partial" if seed_results else "failed"
        failure = {"exception": type(exc).__name__, "message": str(exc), "completed_seeds": sum(x["status"] == "complete" for x in seed_results),
                   "seed_status": statuses, "global_step": global_step,
                   "diagnostic": failure_diagnostic}
        write_json(run_dir / "failure.json", failure)
        manifest.update({"status": status, "seed_status": statuses, "seed_results": seed_results, "failure": failure})
        write_json(run_dir / "summary.json", {"status": status, "seed_results": seed_results,
                    "test_metrics": _test_summary(seed_results),
                    "gradient_norm": {"mean_of_seed_means": statistics.fmean(x["gradient_norm"]["mean"] for x in seed_results),
                                      "max_of_seed_maxima": max(x["gradient_norm"]["max"] for x in seed_results),
                                      "nonfinite_count": sum(x["gradient_norm"]["nonfinite_count"] for x in seed_results)}})
        _write_run_manifest(run_dir, manifest)
        raise


### `metrics.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/metrics.py"
"""Character-weighted losses and diagnostic metrics for training objectives."""
from __future__ import annotations

import torch
import torch.nn.functional as F

from .training_data import valid_initial_final_pairs
from .objectives import parse_model_id

COSINE_EPSILON = 1e-8


def batch_objective(outputs: dict, batch: dict, acoustic_targets: torch.Tensor, model_id: str) -> dict:
    family, weight = parse_model_id(model_id)
    mask = outputs["mask"]
    count = int(mask.sum().item())
    if count <= 0 or not torch.equal(mask, batch["acoustic_ids"] >= 0):
        raise ValueError("batch target masks are empty or misaligned")
    values = {"characters": count, "objective_sum": outputs["hidden"].new_zeros(())}
    if family == "syllable":
        target = batch["syllable_ids"][mask]
        loss = F.cross_entropy(outputs["syllable_logits"][mask], target, reduction="sum")
        values["syllable_ce_sum"] = loss
        values["objective_sum"] = values["objective_sum"] + loss
        values["syllable_correct"] = int((outputs["syllable_logits"][mask].argmax(-1) == target).sum().item())
    if family in {"components", "components-acoustic"}:
        labels = batch["component_labels"][mask]
        ce_sums = []
        for index, name in enumerate(("initials", "finals", "tones")):
            logits = outputs["component_logits"][name][mask]
            target = labels[:, index]
            loss = F.cross_entropy(logits, target, reduction="sum")
            values[f"{name}_ce_sum"] = loss
            values[f"{name}_correct"] = int((logits.argmax(-1) == target).sum().item())
            ce_sums.append(loss)
        component_sum = sum(ce_sums) / 3.0
        values["component_ce_sum"] = component_sum
        values["objective_sum"] = values["objective_sum"] + component_sum
        predicted = torch.stack([outputs["component_logits"][n][mask].argmax(-1) for n in ("initials", "finals")], dim=-1)
        valid_pairs = torch.tensor(sorted(valid_initial_final_pairs()), dtype=torch.long, device=predicted.device)
        valid = (predicted[:, None, :] == valid_pairs[None, :, :]).all(dim=-1).any(dim=-1)
        values["invalid_if_count"] = int((~valid).sum().item())
        values["joint_ift_correct"] = int((torch.stack([outputs["component_logits"][n][mask].argmax(-1) for n in ("initials", "finals", "tones")], dim=-1) == labels).all(dim=-1).sum().item())
    if family in {"acoustic", "components-acoustic"}:
        target_ids = batch["acoustic_ids"][mask]
        prediction = outputs["acoustic_prediction"][mask]
        targets = acoustic_targets[target_ids]
        cosine_distance = 1.0 - F.cosine_similarity(prediction, targets, dim=-1, eps=COSINE_EPSILON)
        cosine_sum = cosine_distance.sum()
        values["cosine_sum"] = cosine_sum
        values["predicted_norm_below_epsilon"] = int((torch.linalg.vector_norm(prediction.float(), dim=-1) < COSINE_EPSILON).sum().item())
        weighted = (weight if family == "components-acoustic" else 1.0) * cosine_sum
        values["weighted_cosine_sum"] = weighted
        values["objective_sum"] = values["objective_sum"] + weighted
    values["objective"] = values["objective_sum"] / count
    return values


def finalize_epoch(totals: dict) -> dict:
    count = int(totals.get("characters", 0))
    if count <= 0:
        raise ValueError("loader contains no valid characters")
    result = {"characters": count, "objective": totals["objective_sum"] / count}
    for key, value in totals.items():
        if key.endswith("_sum") and key != "objective_sum":
            result[key.removesuffix("_sum")] = value / count
    for key in ("syllable_correct", "initials_correct", "finals_correct", "tones_correct", "joint_ift_correct"):
        if key in totals: result[key.removesuffix("_correct") + "_accuracy"] = totals[key] / count
    if "invalid_if_count" in totals:
        result["invalid_if_rate"] = totals["invalid_if_count"] / count
        result["invalid_if_count"] = totals["invalid_if_count"]
    if "predicted_norm_below_epsilon" in totals:
        result["predicted_norm_below_epsilon"] = totals["predicted_norm_below_epsilon"]
    return result


### `training_data.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/training_data.py"
"""Pure Pinyin-inventory helpers shared by training diagnostics."""
from __future__ import annotations

from functools import lru_cache

from .pinyin import VALID_SYLLABLES, canonical_label, decompose


@lru_cache(maxsize=1)
def valid_initial_final_pairs() -> frozenset[tuple[int, int]]:
    """Return pairs represented by labels accepted by the strict Pinyin API."""
    pairs = set()
    for base in sorted(VALID_SYLLABLES):
        try:
            canonical = canonical_label(base + "1", tone_convention="explicit")
            initial_id, final_id, _ = decompose(canonical, tone_convention="explicit")
        except (ValueError, TypeError, IndexError):
            continue
        pairs.add((initial_id, final_id))
    if not pairs:
        raise RuntimeError("strict Pinyin inventory contains no valid initial/final pairs")
    return frozenset(pairs)


### `reproducibility.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/reproducibility.py"
"""Deterministic training seeds and serializable RNG snapshots."""
from __future__ import annotations

import random

import numpy as np
import torch


def seed_everything(seed: int, deterministic: bool = True) -> None:
    if type(seed) is not int or seed < 0:
        raise ValueError("seed must be a nonnegative integer")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.benchmark = False


def make_generator(seed: int, stream: int = 0) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(int(seed) + 1_000_003 * int(stream))
    return generator


def rng_snapshot(loader_generator: torch.Generator | None = None) -> dict:
    numpy_state = np.random.get_state()
    state = {
        "python": list(random.getstate()),
        "numpy": {"algorithm": numpy_state[0], "keys": numpy_state[1].tolist(),
                  "position": int(numpy_state[2]), "has_gauss": int(numpy_state[3]),
                  "cached_gaussian": float(numpy_state[4])},
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }
    if loader_generator is not None:
        state["dataloader"] = loader_generator.get_state()
    return state


### `source_preparation/__init__.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/source_preparation/__init__.py"
"""Pinned DuYu export and safe local sound archive helpers."""

from importlib import import_module

_EXPORTS = {"unpack_sounds": ".archive", "export_duyu": ".duyu",
            "CHINESE_LESSONS_ALIASES": ".aliases",
            "download_sounds": ".download", "validate_source_bundle": ".download"}
__all__ = list(_EXPORTS)


def __getattr__(name):
    if name not in _EXPORTS:
        raise AttributeError(name)
    return getattr(import_module(_EXPORTS[name], __name__), name)


### `source_preparation/archive.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/source_preparation/archive.py"
"""Safe flat sound archive extraction with checksum validation."""

from __future__ import annotations

import hashlib
import os
import shutil
import stat
import tempfile
import zipfile
from pathlib import Path,PurePosixPath,PureWindowsPath

from ..artifacts import sha256_file

ALLOWED={".wav",".mp3"}
WINDOWS_FORBIDDEN=set('<>:"/\\|?*')
WINDOWS_DEVICES={"con","prn","aux","nul",*(f"com{i}" for i in range(1,10)),*(f"lpt{i}" for i in range(1,10))}




def unpack_sounds(archive_path,destination,*,expected_sha256):
    archive_path=Path(archive_path).resolve();destination=Path(destination).resolve()
    if destination.exists(): raise FileExistsError(destination)
    actual_sha=sha256_file(archive_path)
    if actual_sha!=expected_sha256: raise ValueError("archive checksum mismatch")
    with zipfile.ZipFile(archive_path) as archive:
        infos=archive.infolist();_validate_entries(infos)
        inventory={info.filename:hashlib.sha256(archive.read(info)).hexdigest() for info in infos}
        payload={info.filename:archive.read(info) for info in infos}
    parent=destination.parent;parent.mkdir(parents=True,exist_ok=True);temp=Path(tempfile.mkdtemp(prefix=f".{destination.name}.",dir=parent))
    try:
        for name,data in payload.items():
            path=temp/name;path.write_bytes(data)
            if sha256_file(path)!=inventory[name]: raise ValueError("extracted checksum mismatch")
        os.rename(temp,destination)
        return {"path":str(destination),"archive_sha256":actual_sha,"files":len(payload),"bytes":sum(len(x) for x in payload.values()),"inventory":inventory}
    except Exception:
        shutil.rmtree(temp,ignore_errors=True);raise


def _validate_entries(infos):
    if not infos: raise ValueError("sound archive is empty")
    names=[]
    for info in infos:
        name=info.filename
        mode=(info.external_attr>>16)&0xFFFF
        if stat.S_ISLNK(mode): raise ValueError(f"symlink ZIP entry: {name}")
        names.append(name)
    _validate_names(names)


def _validate_names(names):
    for name in names:
        posix=PurePosixPath(name);windows=PureWindowsPath(name)
        device=name.split(".",1)[0].casefold()
        if (not name or name!=posix.name or "\\" in name or posix.is_absolute() or windows.is_absolute()
                or windows.drive or len(posix.parts)!=1 or any(c in WINDOWS_FORBIDDEN for c in name)
                or name.endswith(("."," ")) or device in WINDOWS_DEVICES or Path(name).suffix.lower() not in ALLOWED):
            raise ValueError(f"unsafe or unsupported sound filename: {name}")
    if len(names)!=len(set(names)): raise ValueError("duplicate ZIP entries")
    if len(names)!=len({name.casefold() for name in names}): raise ValueError("case-colliding ZIP entries")


### `source_preparation/download.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/source_preparation/download.py"
"""Pinned, atomic downloads of the official Mandarin sound archive."""
from __future__ import annotations

import os
import json
import shutil
from pathlib import Path
import re
import tempfile
import urllib.request
import zipfile

from ..artifacts import sha256_file, write_json
from .archive import _validate_entries

CHINESE_LESSONS_URL = "https://www.chinese-lessons.com/sounds/Mandarin_sounds.zip"
CHINESE_LESSONS_SHA256 = "be17ce405fcfd6c63a940fa9ac2a7985aa3e6e464543d72f36f2b9dc6ee8b8cd"
# The official server rejects urllib's default User-Agent with HTTP 406.
DOWNLOAD_HEADERS = {"User-Agent": "curl/8.0.0", "Accept": "*/*"}
ARCHIVE_NAME = "Mandarin_sounds.zip"


def validate_source_bundle(path, *, url=CHINESE_LESSONS_URL, expected_sha256=CHINESE_LESSONS_SHA256):
    """Verify complete source content and the requested source identity."""
    root = Path(path).absolute()
    if root.is_symlink() or not root.is_dir():
        raise ValueError("source bundle must be a non-symlink directory")
    if {p.name for p in root.iterdir()} != {ARCHIVE_NAME, "manifest.json"}:
        raise ValueError("source bundle artifact set mismatch")
    manifest_path = root / "manifest.json"
    if manifest_path.is_symlink():
        raise ValueError("source manifest must not be a symlink")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = _source_manifest(url, expected_sha256)
    if manifest != expected:
        raise ValueError("source bundle configuration mismatch; use a new output ID")
    validate_sound_archive(root / ARCHIVE_NAME, expected_sha256)
    return root


def _source_manifest(url, digest):
    return {"schema_version": 1, "bundle_type": "sound_source", "source_url": url,
            "source_page": "https://www.chinese-lessons.com/download.htm",
            "attribution": "Chinese-Lessons.com, Mandarin Sounds",
            "license_url": "https://creativecommons.org/licenses/by-nc-nd/3.0/us/",
            "request_headers": dict(DOWNLOAD_HEADERS), "artifacts": {ARCHIVE_NAME: digest}}


def validate_sound_archive(path, expected_sha256):
    """Validate bytes, safe flat filenames and every entry's CRC."""
    if not isinstance(expected_sha256, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_sha256):
        raise ValueError("expected_sha256 must be a lowercase SHA-256")
    path = Path(path)
    if path.is_symlink() or not path.is_file():
        raise ValueError("archive must be a regular, non-symlink file")
    if sha256_file(path) != expected_sha256:
        raise ValueError("archive checksum mismatch; use a new output for changed inputs")
    with zipfile.ZipFile(path) as archive:
        _validate_entries(archive.infolist())
        if archive.testzip() is not None:
            raise ValueError("archive contains a corrupt entry")
    return path


def download_sounds(destination, *, url=CHINESE_LESSONS_URL,
                    expected_sha256=CHINESE_LESSONS_SHA256, timeout=60):
    """Download or reuse a verified directory containing ZIP and provenance.

    This validates the archive container; decoded-audio checks belong to the
    sound inventory stage. No embeddings or ML dependencies are imported.
    """
    if not isinstance(url, str) or not url.startswith("https://"):
        raise ValueError("sound download requires an HTTPS URL")
    if not isinstance(expected_sha256, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_sha256):
        raise ValueError("expected_sha256 must be a lowercase SHA-256")
    if type(timeout) not in (int, float) or not 0 < timeout <= 3600:
        raise ValueError("timeout must be positive and at most 3600 seconds")
    target = Path(destination).absolute()
    if target.exists() or target.is_symlink():
        return validate_source_bundle(target, url=url, expected_sha256=expected_sha256)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(tempfile.mkdtemp(prefix=f".{target.name}.", suffix=".partial", dir=target.parent))
    try:
        with (temporary / ARCHIVE_NAME).open("wb") as output:
            request = urllib.request.Request(url, headers=DOWNLOAD_HEADERS)
            with urllib.request.urlopen(request, timeout=timeout) as response:
                if response.status != 200 or not response.geturl().startswith("https://"):
                    raise ValueError("download did not return a successful HTTPS response")
                count = 0
                while block := response.read(1024 * 1024):
                    output.write(block)
                    count += len(block)
                declared = response.headers.get("Content-Length")
                if declared is not None and count != int(declared):
                    raise ValueError("incomplete sound archive response")
        write_json(temporary / "manifest.json", _source_manifest(url, expected_sha256))
        validate_source_bundle(temporary, url=url, expected_sha256=expected_sha256)
        if target.exists() or target.is_symlink():
            raise FileExistsError(target)
        os.rename(temporary, target)
        return target
    finally:
        if temporary.exists():
            shutil.rmtree(temporary)


### `source_preparation/duyu.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/source_preparation/duyu.py"
"""Export a pinned DuYu split without generating or guessing pinyin labels."""

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import tempfile
import unicodedata
from collections import Counter
from pathlib import Path

from ..artifacts import sha256_file,write_json,write_jsonl
from ..data_contract import is_han
from ..pinyin import canonical_labels,decompose


def export_duyu(output_dir,*,revision,split,columns,tone_convention,dataset_loader=None,allow_download=False):
    _validate_args(revision,split,columns,tone_convention)
    target=Path(output_dir).resolve()
    if target.exists(): raise FileExistsError(target)
    if dataset_loader is None:
        if not allow_download: raise PermissionError("real DuYu loading requires allow_download=True")
        try: from datasets import load_dataset
        except ImportError as exc: raise RuntimeError("datasets is required for real DuYu loading") from exc
        rows=load_dataset("Duyu/Pinyin-Hanzi",revision=revision,split=split)
        loader_kind="datasets"
    else:
        rows=dataset_loader("Duyu/Pinyin-Hanzi",revision=revision,split=split)
        loader_kind="injected"
    parent=target.parent;parent.mkdir(parents=True,exist_ok=True);temp=Path(tempfile.mkdtemp(prefix=f".{target.name}.",dir=parent))
    accepted=[];rejected=[];raw_rows=[]
    try:
        for index,row in enumerate(rows):
            if not isinstance(row,dict): raise ValueError(f"DuYu row {index} is not an object")
            required={columns["text"],columns["pinyin"]}
            if columns.get("id"): required.add(columns["id"])
            missing=required-set(row)
            if missing: raise ValueError(f"DuYu row {index} is missing mapped columns: {sorted(missing)}")
            raw_text=row.get(columns["text"]); raw_pinyin=row.get(columns["pinyin"]); original_id=row.get(columns.get("id")) if columns.get("id") else None
            source_id=str(original_id) if original_id not in (None,"") else f"{revision}:{split}:{index}"
            base={"source_id":source_id,"source_row":index,"raw_text":raw_text,"raw_pinyin":raw_pinyin}
            cleaned,labels,reason=_normalize(raw_text,raw_pinyin,tone_convention)
            raw_rows.append({**base,"text":cleaned,"labels":labels,"status":"accepted" if reason is None else "rejected","reason":reason})
            if reason: rejected.append({**base,"reason":reason})
            else: accepted.append({"text":cleaned,"pinyin":" ".join(labels),"source_id":source_id})
        with (temp/"corpus.csv").open("w",encoding="utf-8",newline="") as handle:
            writer=csv.DictWriter(handle,fieldnames=["text","pinyin","source_id"]);writer.writeheader();writer.writerows(accepted)
        write_jsonl(temp/"source_rows.jsonl",raw_rows);write_jsonl(temp/"rejections.jsonl",rejected)
        counts={"input":len(raw_rows),"accepted":len(accepted),"rejected":len(rejected),"reasons":dict(sorted(Counter(r["reason"] for r in rejected).items()))}
        if counts["input"]!=counts["accepted"]+counts["rejected"]: raise AssertionError("export count imbalance")
        write_json(temp/"metrics.json",counts)
        files={p.name:sha256_file(p) for p in temp.iterdir() if p.is_file()}
        write_json(temp/"provenance.json",{"schema_version":1,"dataset":"Duyu/Pinyin-Hanzi","revision":revision,"split":split,
            "columns":columns,"tone_convention":tone_convention,"loader":loader_kind,"files":files})
        os.rename(temp,target);return target
    except Exception:
        shutil.rmtree(temp,ignore_errors=True);raise


def _validate_args(revision,split,columns,tone_convention):
    if (not isinstance(revision,str) or not revision.strip() or "REPLACE_WITH" in revision
            or revision.strip().lower() in {"latest","main","master"}): raise ValueError("exact non-placeholder revision required")
    if not isinstance(split,str) or not split.strip() or "REPLACE_WITH" in split: raise ValueError("exact non-placeholder split required")
    if not isinstance(columns,dict) or not {"text","pinyin"}.issubset(columns) or set(columns)-{"text","pinyin","id"}: raise ValueError("explicit text/pinyin columns required")
    if any(not isinstance(value,str) or not value or "REPLACE_WITH" in value for value in columns.values()): raise ValueError("column mapping contains a placeholder")
    if len(set(columns.values()))!=len(columns): raise ValueError("column mappings must be distinct")
    if tone_convention not in {"explicit","neutral_unmarked"}: raise ValueError("unsupported tone convention")


def _normalize(text,pinyin,tone_convention):
    if not isinstance(text,str) or not text: return None,None,"empty"
    if any(ch.isdigit() for ch in text): return None,None,"digit"
    raw=pinyin.split() if isinstance(pinyin,str) else list(pinyin) if isinstance(pinyin,(list,tuple)) else None
    if raw is None or len(text)!=len(raw): return None,None,"alignment"
    kept=[(char,label) for char,label in zip(text,raw) if not char.isspace() and unicodedata.category(char)[0]!="P"]
    cleaned="".join(char for char,_ in kept); labels=[label for _,label in kept]
    if not cleaned: return None,None,"empty_after_cleaning"
    if any(not is_han(char) for char in cleaned): return None,None,"unsupported_character"
    try:
        labels=canonical_labels(labels,tone_convention=tone_convention)
        for label in labels: decompose(label)
    except (ValueError,TypeError,IndexError): return None,None,"pinyin"
    return cleaned,labels,None


### `source_preparation/aliases.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/source_preparation/aliases.py"
"""Canonical labels for the original Chinese Lessons audio filenames."""
CHINESE_LESSONS_ALIASES={
    "lue4.mp3":"lve4","nue4.mp3":"nve4","luu2.mp3":"lv2","luu3.mp3":"lv3",
    "luu4.mp3":"lv4","nuu3.mp3":"nv3","nuu4.mp3":"nv4",
}


### `preparation/__init__.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/preparation/__init__.py"
"""Acoustic inference and model configuration helpers for the character pipeline."""


### `preparation/config.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/preparation/config.py"
"""Acoustic model configuration validation."""
from copy import deepcopy


def validate_model_config(model):
    _keys(model, {"id", "revision", "layer", "pooling", "sample_rate", "mono", "padding", "dtype", "device"}, {"processor"}, "model")
    if not isinstance(model["revision"], str) or not model["revision"]:
        raise ValueError("model revision is required")
    if model["pooling"] not in {"mean_all", "mean_valid"} or model["padding"] not in {"legacy_one_second", "none"}:
        raise ValueError("unsupported pooling or padding")
    if model["pooling"] == "mean_valid":
        raise ValueError("mean_valid is unsupported until padding masks can be proven correct")
    if model["dtype"] != "float32" or not isinstance(model["mono"], bool) or not isinstance(model["layer"], int) or isinstance(model["layer"], bool):
        raise ValueError("model requires dtype=float32, boolean mono, and integer layer")
    if not isinstance(model["device"], str) or not model["device"]:
        raise ValueError("model device must be explicit")
    if not isinstance(model["sample_rate"], int) or isinstance(model["sample_rate"], bool) or model["sample_rate"] <= 0:
        raise ValueError("sample_rate must be a positive integer")
    return deepcopy(model)


def _keys(value, required, optional, name):
    if not required.issubset(value) or set(value) - required - optional:
        raise ValueError(f"invalid {name} keys")


### `preparation/embeddings.py`


In [ ]:
%%writefile -q "{SDK_ROOT}/phonetics/preparation/embeddings.py"
"""Full-inventory acoustic embedding generation with deterministic caching."""

from __future__ import annotations

import hashlib
import json
import pickle
import os
import uuid
import wave
from pathlib import Path

from ..artifacts import sha256_file
from ..data_contract import validate_acoustic


def cache_key(entry, model, backend_identity):
    payload = {"audio_sha256": entry["sha256"], "model": model, "backend": backend_identity}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def embed_inventory(inventory, model_config, cache_root, backend=None):
    import numpy as np

    cache_root = Path(cache_root).resolve()
    cache_root.mkdir(parents=True, exist_ok=True)
    result, computed, cached = {}, 0, 0
    backend_identity = getattr(backend, "cache_id", None) if backend is not None else "transformers"
    use_cache = backend is None or backend_identity is not None
    real_backend = backend
    for entry in inventory:
        key = cache_key(entry, model_config, backend_identity or "uncached-stub")
        cache_path = cache_root / f"{key}.pkl"
        if use_cache and cache_path.is_file():
            with cache_path.open("rb") as handle:
                vector = pickle.load(handle)
            cached += 1
        else:
            if real_backend is None:
                real_backend = TransformersBackend(model_config)
            vector = real_backend(entry, model_config)
            vector = np.asarray(vector, dtype=np.float32)
            validate_acoustic({entry["label"]: vector})
            if use_cache:
                temporary = cache_path.with_name(cache_path.name + f".{uuid.uuid4().hex}.tmp")
                with temporary.open("wb") as handle:
                    pickle.dump(vector, handle, protocol=pickle.HIGHEST_PROTOCOL)
                if cache_path.exists():
                    temporary.unlink()
                else:
                    os.rename(temporary, cache_path)
            computed += 1
        result[entry["label"]] = np.asarray(vector, dtype=np.float32)
    validate_acoustic(result, expected_keys=[row["label"] for row in inventory])
    return result, {"computed": computed, "cached": cached, "errors": 0,
                    "backend": "stub" if backend is not None else "transformers"}


class TransformersBackend:
    def __init__(self, config):
        try:
            import torch
            from transformers import AutoModel, AutoProcessor
        except ImportError as exc:
            raise RuntimeError("torch and transformers are required for real inference") from exc
        self.torch = torch
        processor_id = config.get("processor") or config["id"]
        self.processor = AutoProcessor.from_pretrained(processor_id, revision=config["revision"])
        self.model = AutoModel.from_pretrained(config["id"], revision=config["revision"]).to(config["device"])
        self.model.eval()

    def __call__(self, entry, config):
        import numpy as np
        samples, sample_rate = _read_audio(Path(entry["path"]), mono=config["mono"])
        if sample_rate != config["sample_rate"]:
            try:
                import librosa
            except ImportError as exc:
                raise RuntimeError("librosa is required for resampling") from exc
            samples = librosa.resample(samples, orig_sr=sample_rate, target_sr=config["sample_rate"])
        if config["padding"] == "legacy_one_second" and len(samples) < config["sample_rate"]:
            samples = np.pad(samples, (0, config["sample_rate"] - len(samples)))
        inputs = self.processor(samples, sampling_rate=config["sample_rate"], return_tensors="pt",
                                padding=config["padding"] != "none")
        inputs = {key: value.to(config["device"]) for key, value in inputs.items()}
        with self.torch.inference_mode():
            output = self.model(**inputs, output_hidden_states=True)
        hidden = output.hidden_states[int(config["layer"])]
        if config["pooling"] == "mean_all":
            pooled = hidden.mean(dim=1)
        else:
            if "attention_mask" not in inputs or not hasattr(self.model, "_get_feature_vector_attention_mask"):
                raise ValueError("mean_valid requires model feature-vector attention-mask support")
            mask = self.model._get_feature_vector_attention_mask(hidden.shape[1], inputs["attention_mask"])
            pooled = (hidden * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True)
        return pooled[0].detach().cpu().to(self.torch.float32).numpy()


def _read_audio(path, *, mono):
    import numpy as np
    if path.suffix.lower() == ".wav":
        with wave.open(str(path), "rb") as handle:
            channels, width, rate = handle.getnchannels(), handle.getsampwidth(), handle.getframerate()
            raw = handle.readframes(handle.getnframes())
        if width != 2:
            raise ValueError("stdlib WAV reader supports only 16-bit PCM")
        values = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
        if channels > 1:
            values = values.reshape(-1, channels)
            if mono:
                values = values.mean(axis=1)
            else:
                raise ValueError("multi-channel input requires mono=true")
        return values, rate
    try:
        import librosa
    except ImportError as exc:
        raise RuntimeError("librosa is required for non-WAV audio") from exc
    return librosa.load(str(path), sr=None, mono=mono)


In [ ]:
for relative, expected in SOURCE_HASHES.items():
    actual = hashlib.sha256((SDK_ROOT / "phonetics" / relative).read_text(encoding="utf-8").encode("utf-8")).hexdigest()
    if actual != expected:
        raise ValueError(f"Source snapshot mismatch: {relative}")
sys.path.insert(0, str(SDK_ROOT))
from importlib.metadata import version
for package, expected in (("numpy", "2.1.3"), ("pypinyin", "0.55.0")):
    if version(package) != expected:
        raise RuntimeError(f"{package} {expected} is required")
from phonetics.cli import main


## Configuration

Paths, input pins, sampling and device choices are explicit. Existing outputs are reused only after validation.


In [ ]:
import copy
import json

WORK_ROOT = Path.cwd() / "phonetics-work"
RUN_PREPARATION = False
ALLOW_INFERENCE = False
# Existing vectors must match the pinned acoustic manifest and audio inventory.
ACOUSTIC_INPUT = Path("SET_ACOUSTIC_BUNDLE_PATH").absolute()
CONFIG = {'schema_version': 1, 'paths': {'source': '../datasets/source-data/chinese-lessons-v1', 'export': '../datasets/source-data/duyu-v1', 'inventory': '../datasets/source-data/inventory-v1', 'corpus': '../datasets/processed-corpus/char-v1', 'acoustic': '../datasets/pinyin-embeddings/char-zh-cn-v1', 'prepared': '../datasets/prepared/char-v1', 'experiments': '../datasets/experiments', 'cache': '../datasets/source-data/acoustic-cache', 'runs': '../runs'}, 'corpus': {'revision': '9961d7d05910be7006f2dcfae21f4fc5ce4d560a', 'split': 'train', 'columns': {'text': '我们试试看！', 'pinyin': 'wo3 men shi4 shi4 kan4 ！'}, 'tone_convention': 'neutral_unmarked'}, 'model': {'id': 'jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn', 'processor': 'jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn', 'revision': '99ccb2737be22b8bb50dcfcc39ad4d567fb90cfd', 'device': 'cpu', 'dtype': 'float32', 'layer': -1, 'sample_rate': 16000, 'mono': True, 'padding': 'none', 'pooling': 'mean_all'}, 'acoustic': {'existing_bundle': '../datasets/pinyin-embeddings/duyu-chinese-lessons-zh-cn-v1', 'manifest_sha256': '0280ba452aa1580b7d414a2aa15f05d878bd1be2bf11e56f93133d07495e3641'}, 'experiment_id': 'char-zh-cn-v1', 'experiment': {'schema_version': 4, 'sample_size': 100000, 'seed_count': 5, 'test_seed': 0, 'test_fraction': 0.1, 'validation_fraction': 0.1, 'architecture': {'embedding_dim': 64, 'sentence_bilstm_hidden_per_direction': 64, 'sentence_bilstm_layers': 1, 'context_dim': 128, 'classification_hidden_dim': 64, 'classification_dropout': 0.5, 'acoustic_dim': 1024}, 'training': {'optimizer': 'Adam', 'learning_rate': 0.001, 'weight_decay': 0.0, 'batch_size': 32, 'epochs': 20, 'scheduler': None, 'early_stopping': False, 'gradient_clipping': None, 'cosine_epsilon': 1e-08}, 'model_ids': ['syllable', 'components', 'acoustic', 'components-acoustic-lambda-1', 'components-acoustic-lambda-10']}}
CONFIG = copy.deepcopy(CONFIG)
CONFIG["paths"] = {key: str((WORK_ROOT / key).absolute()) for key in CONFIG["paths"]}
CONFIG["acoustic"]["existing_bundle"] = str(ACOUSTIC_INPUT)
# To compute new vectors, set both acoustic input fields to None and enable ALLOW_INFERENCE.
CONFIG_PATH = WORK_ROOT / "config.json"


In [ ]:
if RUN_PREPARATION:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
    main(['prepare-experiment', "--config", str(CONFIG_PATH)] + (["--allow-inference"] if ALLOW_INFERENCE else []))
else:
    print("Preparation disabled; review the configuration before enabling it.")
